## **PARTIE 1 : Imports et Configuration**

In [1]:
!/usr/bin/env python3
# -*- coding: utf-8 -*-
"""
================================================================================
PIPELINE PFE - OPTIMISATION DES ENCAISSEMENTS ET DU RECOUVREMENT (SEAAL)
================================================================================
Theme : Analyse spatio-temporelle, prediction et segmentation par ML/DL
        appliquees a la detection d'anomalies, a la prevision des
        encaissements/CA/volumes et a la strategie de recouvrement.
Date   : 2025-2026

Tables de donnees :
- Table 1 : Donnees clients/compteurs (principal)
- Table 2 : Plans d'action 2025
- Table 3 : Creances clients
- Table 4 : Chiffre d'affaires 2025
- Table 5 : Encaissements 2025

Objectifs (alignes sur le theme de stage) :
1. Detection et typologie des anomalies via analyse spatio-temporelle et
   geolocalisation (Isolation Forest, LOF, Random Forest supervise et
   reseau de neurones / autoencodeur), avec systeme d'alerte a score de
   risque et cartographie interactive.
2. Prevision automatisee des tendances clients : series temporelles des
   volumes factures, du CA et des encaissements (modeles statistiques,
   Machine Learning et reseaux de neurones), pipeline automatise et
   tableaux de bord d'evaluation des erreurs.
3. Segmentation client strategique par analyse multidimensionnelle pour
   piloter des strategies de recouvrement differenciees et en suivre
   l'efficacite.
================================================================================
"""

# [FIX] La cellule d'origine appelait `print(raw.columns.tolist())` avant
# meme le chargement des donnees (variable `raw` jamais definie) -> NameError
# systematique. Cette instruction de debogage a ete supprimee. Pour inspecter
# les colonnes brutes, utiliser `print(df.columns.tolist())` juste apres
# `DataLoader(...).load_all()` (PARTIE 2, en fin de fichier).

import warnings
warnings.filterwarnings('ignore')

import os
import sys
import json
import pickle
import logging
import subprocess
from datetime import datetime, timedelta
from typing import Dict, List, Tuple, Optional, Union
from dataclasses import dataclass, field

import numpy as np
import pandas as pd
pd.set_option('display.max_columns', 50)
pd.set_option('display.width', 200)

import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import matplotlib.dates as mdates
import seaborn as sns

from scipy import stats
from scipy.spatial.distance import cdist
from scipy.spatial import cKDTree
from scipy.cluster.hierarchy import dendrogram, linkage

from sklearn.preprocessing import StandardScaler, MinMaxScaler, RobustScaler, LabelEncoder
from sklearn.impute import SimpleImputer, KNNImputer
from sklearn.decomposition import PCA
from sklearn.cluster import KMeans, DBSCAN, AgglomerativeClustering
from sklearn.mixture import GaussianMixture
from sklearn.ensemble import (
    IsolationForest, RandomForestRegressor, RandomForestClassifier,
    GradientBoostingRegressor
)
from sklearn.svm import OneClassSVM, SVR
from sklearn.neighbors import LocalOutlierFactor, KNeighborsRegressor
from sklearn.linear_model import LogisticRegression, Ridge
from sklearn.neural_network import MLPRegressor, MLPClassifier
from sklearn.metrics import (
    mean_squared_error, mean_absolute_error, r2_score,
    silhouette_score, calinski_harabasz_score, davies_bouldin_score,
    accuracy_score, precision_score, recall_score, f1_score,
    roc_auc_score, confusion_matrix, classification_report
)
from sklearn.model_selection import train_test_split, cross_val_score, GridSearchCV, TimeSeriesSplit

# --- TensorFlow / Keras (reseaux de neurones - optionnel) ------------------
import os
os.environ['PYTHONHASHSEED'] = '0'
os.environ['TF_DETERMINISTIC_OPS'] = '1'
os.environ['TF_CUDNN_DETERMINISTIC'] = '1'
os.environ['TF_NUM_INTRAOP_THREADS'] = '1'
os.environ['TF_NUM_INTEROP_THREADS'] = '1'


try:
    import tensorflow as tf
    from tensorflow.keras.models import Sequential, Model
    from tensorflow.keras.layers import Dense, LSTM, Dropout, BatchNormalization, Input, RepeatVector, TimeDistributed
    from tensorflow.keras.callbacks import EarlyStopping, ReduceLROnPlateau
    from tensorflow.keras.optimizers import Adam
    TF_AVAILABLE = True
except ImportError:
    TF_AVAILABLE = False
    print("[!] TensorFlow non installe. Repli automatique sur MLP scikit-learn "
          "pour les reseaux de neurones (autoencodeur / prediction).")

try:
    tf.config.threading.set_intra_op_parallelism_threads(1)
    tf.config.threading.set_inter_op_parallelism_threads(1)
except RuntimeError:
    logger.warning("Configuration threading TensorFlow deja verrouillee (kernel deja initialise) — ignoree.")

# --- XGBoost / LightGBM -----------------------------------------------------
try:
    from xgboost import XGBRegressor, XGBClassifier
    XGB_AVAILABLE = True
except ImportError:
    XGB_AVAILABLE = False

try:
    from lightgbm import LGBMRegressor, LGBMClassifier
    LGB_AVAILABLE = True
except ImportError:
    LGB_AVAILABLE = False

# --- statsmodels (modeles statistiques de series temporelles) --------------
try:
    from statsmodels.tsa.holtwinters import ExponentialSmoothing
    from statsmodels.tsa.arima.model import ARIMA
    from statsmodels.tsa.seasonal import seasonal_decompose
    STATSMODELS_AVAILABLE = True
except ImportError:
    STATSMODELS_AVAILABLE = False
    print("[!] statsmodels non installe. Le lissage exponentiel / ARIMA sera "
          "remplace par une moyenne mobile ponderee (repli statistique).")

# --- Folium (cartographie) --------------------------------------------------
try:
    import folium
    from folium.plugins import MarkerCluster, HeatMap, AntPath
    from folium import CircleMarker, Popup
    FOLIUM_AVAILABLE = True
except ImportError:
    FOLIUM_AVAILABLE = False

# --- Geopy (distances geodesiques) ------------------------------------------
try:
    from geopy.distance import geodesic
    GEOPY_AVAILABLE = True
except ImportError:
    GEOPY_AVAILABLE = False

# Configuration graphique
plt.style.use('seaborn-v0_8-whitegrid')
sns.set_palette("husl")

RANDOM_STATE = 42
np.random.seed(RANDOM_STATE)
if TF_AVAILABLE:
    tf.random.set_seed(RANDOM_STATE)

OUTPUT_DIR = "/mnt/agents/output"
os.makedirs(OUTPUT_DIR, exist_ok=True)
os.makedirs(os.path.join(OUTPUT_DIR, "models"), exist_ok=True)

logging.basicConfig(
    level=logging.INFO,
    format='%(asctime)s [%(levelname)s] %(message)s',
    handlers=[
        logging.FileHandler(os.path.join(OUTPUT_DIR, 'pipeline.log')),
        logging.StreamHandler(sys.stdout)
    ]
)
logger = logging.getLogger(__name__)

print("=" * 70)
print("PIPELINE PFE - Optimisation des Encaissements et du Recouvrement")
print("=" * 70)
print(f"Repertoire de sortie : {OUTPUT_DIR}")
print(f"TensorFlow    : {'OK' if TF_AVAILABLE else 'NON DISPONIBLE (repli MLP sklearn)'}")
print(f"XGBoost       : {'OK' if XGB_AVAILABLE else 'NON DISPONIBLE'}")
print(f"LightGBM      : {'OK' if LGB_AVAILABLE else 'NON DISPONIBLE'}")
print(f"statsmodels   : {'OK' if STATSMODELS_AVAILABLE else 'NON DISPONIBLE (repli moyenne mobile)'}")
print(f"Folium        : {'OK' if FOLIUM_AVAILABLE else 'NON DISPONIBLE'}")
print(f"Geopy         : {'OK' if GEOPY_AVAILABLE else 'NON DISPONIBLE'}")
print("=" * 70)

The system cannot find the path specified.


PIPELINE PFE - Optimisation des Encaissements et du Recouvrement
Repertoire de sortie : /mnt/agents/output
TensorFlow    : OK
XGBoost       : OK
LightGBM      : OK
statsmodels   : OK
Folium        : OK
Geopy         : OK


## **PARTIE 1bis : Installation des dependances optionnelles**

[FIX] La cellule d'origine utilisait la commande magique IPython
`!{sys.executable} -m pip install ...`. Cette syntaxe casse tout script
execute hors notebook (`python pipeline.py`) car `!` n'est pas du Python
valide. On la remplace par un appel `subprocess` portable, encadre par un
`try/except` pour ne jamais faire planter le pipeline si le reseau/pip
n'est pas disponible (l'environnement fonctionne alors en mode degrade
grace aux drapeaux *_AVAILABLE ci-dessus).

In [2]:
def installer_dependances_optionnelles():
    """Installe les paquets optionnels manquants (best-effort, non bloquant)."""
    paquets = []
    if not XGB_AVAILABLE:
        paquets.append("xgboost")
    if not LGB_AVAILABLE:
        paquets.append("lightgbm")
    if not GEOPY_AVAILABLE:
        paquets.append("geopy")
    if not STATSMODELS_AVAILABLE:
        paquets.append("statsmodels")
    if not FOLIUM_AVAILABLE:
        paquets.append("folium")

    if not paquets:
        logger.info("Toutes les dependances optionnelles sont deja disponibles.")
        return

    logger.info(f"Installation des paquets manquants : {paquets}")
    try:
        subprocess.run(
            [sys.executable, "-m", "pip", "install", "--quiet", *paquets],
            check=True, timeout=600
        )
        logger.info("Installation terminee. Redemarrer le kernel pour charger "
                     "les nouveaux modules si necessaire.")
    except Exception as e:
        logger.warning(f"Installation optionnelle echouee ({e}). "
                        "Le pipeline continue en mode degrade.")


# Decommenter pour lancer l'installation automatique :
# installer_dependances_optionnelles()

## **PARTIE 2 : Configuration et Chargement des Données**

In [3]:
# ==============================================================================
# PARTIE 2 : CONFIGURATION ET CHARGEMENT DES DONNEES
# ==============================================================================

@dataclass
class PipelineConfig:
    """Configuration globale du pipeline (instance unique partagee - voir FIX plus bas)."""
    # Chemins des fichiers
    table1_path: str = "math/T1 2026 au T3 2025 - Copie 2.xlsx"      # Donnees principales
    table2_path: str = "math/PDA 2025 Sidi abdellah V.xlsx"          # Plans d'action
    table3_path: str = "math/créances sidi abdellah.xlsx"           # Creances
    table4_path: str = "math/CA sidi abdellah.xlsx"                 # CA 2025
    table5_path: str = "math/Encaissement 2025 Sidi Abdellah V.xlsx" # Encaissements

    output_path: str = OUTPUT_DIR
    models_path: str = os.path.join(OUTPUT_DIR, "models")

    # Parametres metier
    taux_accessibilite_min: float = 0.7   # Seuil accessibilite critique
    indice_criticite_max: float = 0.8     # Seuil criticite haute
    ecart_conso_max: float = 3.0          # Seuil instabilite conso

    # Parametres recouvrement / encaissement (theme point 2 et 3)
    seuil_creance_critique: float = 1000.0
    seuil_taux_encaissement_bas: float = 0.5
    horizon_prevision_mois: int = 3       # Horizon de prevision des encaissements/CA
    n_clusters_defaut: int = 5

    # ML
    contamination: float = 0.05
    n_clusters: int = 5
    test_size: float = 0.2

    def __post_init__(self):
        os.makedirs(self.models_path, exist_ok=True)


class DataLoader:
    """Charge et fusionne les 5 tables de donnees."""

    def __init__(self, config: PipelineConfig):
        self.config = config
        self.tables = {}
        self.encaissements_bruts: Optional[pd.DataFrame] = None  # [NEW] conserve la
        # granularite transaction (date x montant) necessaire pour la
        # prevision de series temporelles de la PARTIE 6 - avant, seule la
        # version agregee par client (perte totale de la dimension date)
        # etait conservee, ce qui rendait toute prevision temporelle
        # impossible avec le pipeline d'origine.

    def load_all(self) -> pd.DataFrame:
        """Charge toutes les tables et cree le dataset unifie."""
        logger.info("=" * 60)
        logger.info("CHARGEMENT DES DONNEES")
        logger.info("=" * 60)

        # Table 1 : Donnees principales (client/compteur)
        df = self._load_table1()

        # Jointures avec les tables annexes
        df = self._join_table2(df)  # Plans d'action
        df = self._join_table3(df)  # Creances
        df = self._join_table4(df)  # CA 2025
        df = self._join_table5(df)  # Encaissements

        logger.info(f"Dataset final : {df.shape[0]} lignes, {df.shape[1]} colonnes")
        return df

    def _load_table1(self) -> pd.DataFrame:
        """Charge la table principale avec les donnees client/compteur."""
        logger.info(f"Chargement Table 1 : {self.config.table1_path}")
        df = pd.read_excel(self.config.table1_path)

        # ['CODE', 'adresse_pntcpg', 'A1', 'A2', 'X_DEPOT', 'Y_DEPOT', 'IDTTRNTYP', 'IDTCYCFAC',
        # 'IDTCTR', 'USAGES', 'Dernier type de facturation', 'dmt', 'DATLCTDRN', 'DATLCTPRC',
        # 'jour RLV', 'ANNE CPR', 'code_obs', 'CNS_FAC']

        # Convert to date type
        df['DATLCTDRN'] = pd.to_datetime(
            df['DATLCTDRN'],
            origin='1899-12-30',
            unit='D'
        )

        df['DATLCTPRC'] = pd.to_datetime(
            df['DATLCTPRC'],
            origin='1899-12-30',
            unit='D'
        )


        # Transformation
        df.loc[df['Dernier type de facturation'] == 'REEL', 'code_obs'] = 'releve'


        rename_map = {
            'CODE': 'code_client',
            'IDTCTR': 'id_controle',
            'A1': 'x_client',
            'A2': 'y_client',
            'X_DEPOT': 'x_depot',
            'Y_DEPOT': 'y_depot',
            'IDTTRNTYP': 'code_tournee',
            'IDTCYCFAC': 'cycle_facturation',
            'USAGES': 'type_usage',
            'dmt': 'diametre_compteur',
            'ANNE CPR': 'age_compteur',
            'CNS_FAC': 'volume_facture',
            'code_obs': 'code_observation',
            'Temps de service': 'temps_service',       # None
            "Taux d'accessibilité": 'taux_accessibilite',       # None
            'Matrice de Temps': 'temps_trajet',        # None
            "Indice d'Attractivité": 'indice_attractivite',     # None
            'Écart de Conso': 'ecart_consommation',    # None
            'Densité Locale': 'densite_compteurs_km',  # None
            'Indice Criticité': 'indice_criticite',    # None
            'IDTAGT': 'id_agent',
            'Fiabilité Agent': 'fiabilite_agent',      # None
            "Taux d'Accès": 'taux_acces_zone',         # None
            'Variabilité Temps': 'variabilite_temps',  # None
            'Indice de Proximité': 'indice_proximite', # None
            'Dernier type de facturation': 'type_facturation'
        }


        df = df.rename(columns={k: v for k, v in rename_map.items() if k in df.columns})

        # Conversion types
        df['code_client'] = df['code_client'].astype(str)
        for cat_col in ['cycle_facturation', 'type_usage', 'type_facturation']:
            if cat_col in df.columns:
                df[cat_col] = df[cat_col].astype('category')

        # Coordonnees GPS
        for col in ['x_client', 'y_client', 'x_depot', 'y_depot']:
            if col in df.columns:
                df[col] = pd.to_numeric(df[col], errors='coerce')

        logger.info(f"Table 1 : {df.shape[0]} clients, {df.shape[1]} variables")
        return df

    def _join_table2(self, df: pd.DataFrame) -> pd.DataFrame:
        """Joint la table des plans d'action 2025."""
        logger.info("Jointure Table 2 : Plans d'action")

        t2 = pd.read_excel(self.config.table2_path)
        t2 = t2.rename(columns={'IDTCLT': 'code_client', 'DATE_PDA': 'date_plan_action', 'LIB': 'libelle_action'})
        t2['code_client'] = t2['code_client'].astype(str)

        # One-hot encoding des types d'action
        action_dummies = pd.get_dummies(t2['libelle_action'].astype(str).str.strip(), prefix='action')
        t2 = pd.concat([t2[['code_client', 'date_plan_action']], action_dummies], axis=1)

        # [FIX] Bug bloquant : le code d'origine agregeait sur des noms de
        # colonnes codes en dur ('action_Coupures', 'action_Envoi E-mail',
        # 'action_Juridique'). Si le fichier source ne contient pas EXACTEMENT
        # ces libelles (orthographe, espace, nouvelle categorie d'action...),
        # `t2_agg = t2.groupby(...).agg({...})` levait un KeyError et
        # arretait tout le pipeline. On construit desormais le dictionnaire
        # d'agregation dynamiquement a partir des colonnes reellement
        # generees par get_dummies, ce qui rend le chargement robuste a
        # n'importe quel jeu de libelles d'action.
        action_cols = [c for c in t2.columns if c.startswith('action_')]
        agg_dict = {'date_plan_action': 'max'}
        agg_dict.update({c: 'sum' for c in action_cols})

        t2_agg = t2.groupby('code_client').agg(agg_dict).reset_index()
        t2_agg['a_plan_action'] = 1  # Flag

        df = df.merge(t2_agg, on='code_client', how='left')
        df['a_plan_action'] = df['a_plan_action'].fillna(0).astype(int)
        for col in action_cols:
            df[col] = df[col].fillna(0).astype(int)

        logger.info(f"  Plans d'action joints : {df['a_plan_action'].sum()} clients concernes "
                     f"({len(action_cols)} types d'action detectes)")
        return df

    def _join_table3(self, df: pd.DataFrame) -> pd.DataFrame:
        """Joint la table des creances."""
        logger.info("Jointure Table 3 : Creances")

        t3 = pd.read_excel(self.config.table3_path)
        t3 = t3.rename(columns={'IDTCLT': 'code_client', 'SLD02': 'creance_totale'})
        t3['code_client'] = t3['code_client'].astype(str)
        t3['creance_totale'] = pd.to_numeric(t3['creance_totale'], errors='coerce').fillna(0)

        # Categories de creance
        t3['niveau_creance'] = pd.cut(
            t3['creance_totale'],
            bins=[-float('inf'), 0, 100, 500, 1000, float('inf')],
            labels=['Aucune', 'Faible', 'Moyenne', 'Elevee', 'Critique']
        )

        df = df.merge(t3[['code_client', 'creance_totale', 'niveau_creance']], on='code_client', how='left')
        df['creance_totale'] = df['creance_totale'].fillna(0)
        # [FIX] pd.cut produit une Categorical : fillna avec une valeur absente
        # des categories ('Aucune' existe deja, donc OK ici) - mais on
        # s'assure explicitement que 'Aucune' fait bien partie des categories
        # avant le fillna pour eviter un ValueError silencieux sur d'autres
        # jeux de bins.
        if 'Aucune' not in df['niveau_creance'].cat.categories:
            df['niveau_creance'] = df['niveau_creance'].cat.add_categories(['Aucune'])
        df['niveau_creance'] = df['niveau_creance'].fillna('Aucune')

        logger.info(f"  Creance moyenne : {df['creance_totale'].mean():.2f}")
        return df

    def _join_table4(self, df: pd.DataFrame) -> pd.DataFrame:
        """Joint le chiffre d'affaires 2025."""
        logger.info("Jointure Table 4 : CA 2025")

        t4 = pd.read_excel(self.config.table4_path)
        t4 = t4.rename(columns={'IDTCLT': 'code_client', 'TTC': 'ca_2025'})
        t4['code_client'] = t4['code_client'].astype(str)
        t4['ca_2025'] = pd.to_numeric(t4['ca_2025'], errors='coerce').fillna(0)

        df = df.merge(t4[['code_client', 'ca_2025']], on='code_client', how='left')
        df['ca_2025'] = df['ca_2025'].fillna(0)

        # [NEW] Conserve la table CA a la granularite date si une colonne de
        # date est presente (necessaire au theme point 2 : series
        # temporelles du CA). On tente plusieurs noms de colonnes courants.
        date_col_ca = next((c for c in ['DATFAC', 'DAT_FACTURATION', 'DATE_FACTURATION', 'DATCRT']
                             if c in t4.columns), None)
        if date_col_ca:
            self.tables['ca_serie'] = t4.rename(columns={date_col_ca: 'date_ca'})[
                ['code_client', 'date_ca', 'ca_2025']
            ].copy()

        logger.info(f"  CA total 2025 : {df['ca_2025'].sum():.2f}")
        return df

    def _join_table5(self, df: pd.DataFrame) -> pd.DataFrame:
        """Joint les encaissements 2025."""
        logger.info("Jointure Table 5 : Encaissements")

        t5 = pd.read_excel(self.config.table5_path)
        t5 = t5.rename(columns={'IDTCLT': 'code_client', 'DATCRT': 'date_encaissement', 'MONTANT': 'montant_encaisse'})
        t5['code_client'] = t5['code_client'].astype(str)
        t5['montant_encaisse'] = pd.to_numeric(t5['montant_encaisse'], errors='coerce').fillna(0)
        t5['date_encaissement'] = pd.to_datetime(t5['date_encaissement'], errors='coerce')

        # [NEW] Sauvegarde la version transactionnelle (une ligne par
        # encaissement, avec sa date) AVANT l'agregation par client.
        # C'est cette serie qui alimente la PARTIE 6 (prevision temporelle).
        self.encaissements_bruts = t5[['code_client', 'date_encaissement', 'montant_encaisse']].dropna(
            subset=['date_encaissement']
        ).copy()

        # Agregation par client (vue "recouvrement" utilisee par la PARTIE 7)
        t5_agg = t5.groupby('code_client').agg({
            'montant_encaisse': 'sum',
            'date_encaissement': 'max'  # Dernier encaissement
        }).reset_index()
        # [FIX] `(pd.Timestamp.now() - NaT).dt.days` renvoie NaN proprement,
        # mais pd.Timestamp.now() est naive alors que des dates Excel
        # peuvent revenir tz-aware selon la source -> TypeError potentiel.
        # On force la comparaison en naive des deux cotes.
        # now_naive = pd.Timestamp.now().tz_localize(None)
        # date_naive = t5_agg['date_encaissement'].dt.tz_localize(None) if pd.api.types.is_datetime64tz_dtype(
        #     t5_agg['date_encaissement']) else t5_agg['date_encaissement']
        # t5_agg['delai_dernier_encaissement'] = (now_naive - date_naive).dt.days

        # df = df.merge(t5_agg, on='code_client', how='left')
        # df['montant_encaisse'] = df['montant_encaisse'].fillna(0)
        # df['delai_dernier_encaissement'] = df['delai_dernier_encaissement'].fillna(999)

        
        # Merge avec le DataFrame principal
        
        df = df.merge(t5_agg, on='code_client', how='left')

        df['delai_dernier_encaissement'] = (df['date_encaissement'] - df['DATLCTDRN']).dt.days



        # Fill NaN values for clients without encaissements 

        df['montant_encaisse'] = df['montant_encaisse'].fillna(0)

        # Fill NaN values for delai_dernier_encaissement with a large number (e.g., 999) but not 0 to indicate no encaissement

        df['delai_dernier_encaissement'] = df['delai_dernier_encaissement'].fillna(999)

        # Remplacer les valeurs négatives par 0

        df.loc[df['delai_dernier_encaissement'] < 0, 'delai_dernier_encaissement'] = 0

        # Taux d'encaissement

        df['taux_encaissement'] = (df['montant_encaisse'] / df['ca_2025'].replace(0, np.nan)).fillna(0).clip(0, 2)

        logger.info(f"  Taux d'encaissement moyen : {df['taux_encaissement'].mean():.2%}")
        return df

In [4]:
# [FIX] Chargement : une SEULE instance de PipelineConfig partagee par tout
# le pipeline. Le notebook d'origine recreait `PipelineConfig()` a chaque
# cellule (PARTIE 2, 3, 5, 6, 7...) : sans consequence tant que la config
# reste par defaut, mais source de bugs des qu'un chemin ou un
# hyper-parametre est modifie en cours de session (les autres etapes
# continuaient de lire l'ancienne config par defaut).
CONFIG = PipelineConfig()

loader = DataLoader(CONFIG)
df_origin = loader.load_all()
print(df_origin.columns.tolist())

2026-09-24 10:01:10,606 [INFO] ============================================================
2026-09-24 10:01:10,606 [INFO] CHARGEMENT DES DONNEES
2026-09-24 10:01:10,606 [INFO] ============================================================
2026-09-24 10:01:10,606 [INFO] Chargement Table 1 : math/T1 2026 au T3 2025 - Copie 2.xlsx
2026-09-24 10:02:12,002 [INFO] Table 1 : 251348 clients, 19 variables
2026-09-24 10:02:12,002 [INFO] Jointure Table 2 : Plans d'action
2026-09-24 10:02:26,359 [INFO]   Plans d'action joints : 189997 clients concernes (11 types d'action detectes)
2026-09-24 10:02:26,375 [INFO] Jointure Table 3 : Creances
2026-09-24 10:02:32,028 [INFO]   Creance moyenne : 4873.61
2026-09-24 10:02:32,038 [INFO] Jointure Table 4 : CA 2025
2026-09-24 10:02:37,001 [INFO]   CA total 2025 : 806346987.88
2026-09-24 10:02:37,007 [INFO] Jointure Table 5 : Encaissements
2026-09-24 10:02:51,133 [INFO]   Taux d'encaissement moyen : 96.95%
2026-09-24 10:02:51,143 [INFO] Dataset final : 251348 l

In [5]:
df_origin.columns

Index(['code_client', 'id_agent', 'adresse_pntcpg', 'x_client', 'y_client', 'x_depot', 'y_depot', 'code_tournee', 'cycle_facturation', 'id_controle', 'type_usage', 'type_facturation',
       'diametre_compteur', 'DATLCTDRN', 'DATLCTPRC', 'jour RLV', 'age_compteur', 'code_observation', 'volume_facture', 'date_plan_action', 'action_Coupures', 'action_Envoi E-mail',
       'action_Juridique', 'action_Mise en Demeure', 'action_Relance Avocat', 'action_Relance Gestion Personnalisée', 'action_Relance Porte à porte', 'action_Relance SMS Exigibilité',
       'action_Relance Systématique', 'action_Relancer le client via CATO', 'action_Télé-Relance', 'a_plan_action', 'creance_totale', 'niveau_creance', 'ca_2025', 'montant_encaisse',
       'date_encaissement', 'delai_dernier_encaissement', 'taux_encaissement'],
      dtype='object')

In [6]:
df = df_origin.copy()  # Travail sur une copie pour ne pas modifier l'original

## **PARTIE 3 : Nettoyage et Feature Engineering Spécifique**

**The Route's Bouding Box:**

          max_lat
             ↑
    ┌─────────────────┐
    │  •     •        │
    │       •         │
    │   •        •    │
    └─────────────────┘
min_lon             max_lon
             ↓
          min_lat

In [7]:
# ==============================================================================
# PARTIE 3 : NETTOYAGE ET FEATURE ENGINEERING
# ==============================================================================
from sklearn.cluster import MiniBatchKMeans

class DataCleaner:
    """Nettoyage specifique aux donnees clients/compteurs/recouvrement."""

    def __init__(self, config: PipelineConfig):
        self.config = config

    def clean(self, df: pd.DataFrame) -> pd.DataFrame:
        """Pipeline de nettoyage complet."""
        logger.info("=" * 60)
        logger.info("NETTOYAGE DES DONNEES")
        logger.info("=" * 60)

        df = df.copy()
        n_before = len(df)

        # Suppression des doublons
        df = df.drop_duplicates(subset=['code_client'], keep='first')

        # Coordonnees GPS invalides
        for col in ['x_client', 'y_client']:
            if col in df.columns:
                df.loc[df[col].abs() > 180, col] = np.nan

        # Imputation GPS par mediane par tournee, avec repli global
        # [FIX] Si `code_tournee` est absent ou si un groupe est integralement
        # NaN, `transform(lambda x: x.fillna(x.median()))` renvoie NaN pour
        # tout le groupe -> des NaN residuels remontaient jusqu'aux features
        # spatiales (KDTree, DBSCAN) et faisaient planter le clustering.
        # On ajoute un repli sur la mediane globale.
        for coord in ['x_client', 'y_client']:
            if coord in df.columns and df[coord].isnull().sum() > 0:
                if 'code_tournee' in df.columns:
                    df[coord] = df.groupby('code_tournee')[coord].transform(
                        lambda x: x.fillna(x.median())
                    )
                df[coord] = df[coord].fillna(df[coord].median())



        #+++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++
        
        # =============================================================================
        # CONTROLE ET CORRECTION DES POSITIONS GPS
        # =============================================================================
        #
        # Objectif :
        #   - d�tecter les clients g�ographiquement incoh�rents
        #   - d�tecter les clients �loign�s de leur tourn�e
        #   - d�tecter les clients hors commune (si fichier communes fourni)
        #   - proposer une position corrig�e
        #   - NE JAMAIS �craser les coordonn�es originales
        #
        # Colonnes attendues :
        #   x_client       = longitude
        #   y_client       = latitude
        #   code_tournee   = tourn�e du client
        #   code_client    = identifiant client
        #
        # =============================================================================



        # =============================================================================
        # PARAMETRES
        # =============================================================================

        GPS_CONFIG = {
            # Distance maximale entre un client et ses voisins de m�me tourn�e
            # avant de consid�rer la position comme suspecte.
            "distance_max_tournee_km": 2.0,

            # Distance minimale pour consid�rer qu'un client est beaucoup
            # plus proche d'une autre tourn�e que de sa propre tourn�e.
            "distance_changement_tournee_km": 3.0,

            # Nombre minimum de clients valides n�cessaires dans une tourn�e
            # pour calculer une r�f�rence fiable.
            "min_clients_tournee": 5,

            # Nombre de voisins utilis�s pour proposer une correction.
            "k_voisins_correction": 5,

            # Une correction automatique n'est appliqu�e que si la confiance
            # est au-dessus de ce seuil.
            "seuil_confiance_correction": 0.80,

            # IMPORTANT :
            # False = on cr�e seulement une proposition de correction.
            # True  = on applique les corrections � x_client/y_client.
            "APPLIQUER_CORRECTIONS": False,

            # Fichier SIG optionnel des communes.
            #
            # Exemple :
            # "/chemin/vers/communes_algerie.shp"
            #
            # Si None -> contr�le hors commune d�sactiv�.
            "fichier_communes": None,

            # Nom de la colonne contenant le nom/code de commune
            # dans le shapefile.
            "colonne_commune": "NOM"
        }


        # =============================================================================
        # DISTANCE GPS RAPIDE EN KM
        # =============================================================================

        def distance_km_vecteur(x1, y1, x2, y2):
            """
            Distance approximative en km entre coordonn�es GPS.
            Adapt�e aux distances locales en Alg�rie.
            """

            lat_ref = np.deg2rad(
                np.nanmean(
                    np.asarray(y1, dtype=float)
                )
            )

            dx = (
                np.asarray(x1, dtype=float)
                - np.asarray(x2, dtype=float)
            ) * np.cos(lat_ref)

            dy = (
                np.asarray(y1, dtype=float)
                - np.asarray(y2, dtype=float)
            )

            return np.sqrt(dx ** 2 + dy ** 2) * 111.0


        # =============================================================================
        # CONTROLE DES COORDONNEES
        # =============================================================================

        def controler_positions_clients(
            df,
            config=GPS_CONFIG
        ):
            """
            D�tecte les incoh�rences g�ographiques et propose une correction.

            Le DataFrame original n'est pas modifi� sauf si
            APPLIQUER_CORRECTIONS=True.
            """

            df = df.copy()

            logger.info("=" * 70)
            logger.info("CONTROLE DES POSITIONS GPS CLIENTS")
            logger.info("=" * 70)

            # -------------------------------------------------------------------------
            # V�rification des colonnes
            # -------------------------------------------------------------------------

            colonnes_obligatoires = [
                "x_client",
                "y_client"
            ]

            for col in colonnes_obligatoires:
                if col not in df.columns:
                    raise ValueError(
                        f"Colonne obligatoire absente : {col}"
                    )

            # -------------------------------------------------------------------------
            # Sauvegarde des coordonn�es originales
            # -------------------------------------------------------------------------

            df["x_client_original"] = pd.to_numeric(
                df["x_client"],
                errors="coerce"
            )

            df["y_client_original"] = pd.to_numeric(
                df["y_client"],
                errors="coerce"
            )

            # -------------------------------------------------------------------------
            # Initialisation des colonnes de contr�le
            # -------------------------------------------------------------------------

            df["gps_anomalie"] = False

            df["gps_anomalie_type"] = "OK"

            df["gps_distance_tournee_km"] = np.nan

            df["gps_tournee_proche"] = np.nan

            df["gps_distance_tournee_proche_km"] = np.nan

            df["gps_commune"] = np.nan

            df["gps_hors_commune"] = False

            df["x_client_corrige"] = df["x_client_original"]

            df["y_client_corrige"] = df["y_client_original"]

            df["gps_correction_proposee"] = False

            df["gps_confiance_correction"] = 0.0

            df["gps_source_correction"] = "Aucune"

            # -------------------------------------------------------------------------
            # Coordonn�es valides
            # -------------------------------------------------------------------------

            valid_gps = (
                df["x_client_original"].between(-180, 180)
                &
                df["y_client_original"].between(-90, 90)
            )

            valid_gps &= (
                df["x_client_original"].notna()
                &
                df["y_client_original"].notna()
            )

            logger.info(
                f"Clients avec GPS valide : "
                f"{valid_gps.sum():,}/{len(df):,}"
            )

            # =============================================================================
            # 1. CONTROLE PAR TOURNEE
            # =============================================================================

            if "code_tournee" in df.columns:

                logger.info("Analyse de coh�rence g�ographique par tourn�e...")

                # -------------------------------------------------------------------------
                # R�f�rences par tourn�e
                # -------------------------------------------------------------------------

                refs_tournees = (
                    df.loc[valid_gps]
                    .groupby("code_tournee")
                    .agg(
                        x_centre=("x_client_original", "median"),
                        y_centre=("y_client_original", "median"),
                        nb_clients=("x_client_original", "size")
                    )
                )

                refs_tournees = refs_tournees[
                    refs_tournees["nb_clients"]
                    >= config["min_clients_tournee"]
                ]

                # -------------------------------------------------------------------------
                # Distance du client au centre de sa tourn�e
                # -------------------------------------------------------------------------

                for idx in df.index[valid_gps]:

                    tournee = df.at[idx, "code_tournee"]

                    if tournee not in refs_tournees.index:
                        continue

                    ref = refs_tournees.loc[tournee]

                    distance = distance_km_vecteur(
                        df.at[idx, "x_client_original"],
                        df.at[idx, "y_client_original"],
                        ref["x_centre"],
                        ref["y_centre"]
                    )

                    df.at[idx, "gps_distance_tournee_km"] = float(distance)

                # -------------------------------------------------------------------------
                # Recherche d'une autre tourn�e g�ographiquement plus coh�rente
                # -------------------------------------------------------------------------

                refs_valides = refs_tournees.dropna()

                if len(refs_valides) >= 2:

                    coords_tournees = np.column_stack([
                        refs_valides["x_centre"].values,
                        refs_valides["y_centre"].values
                    ])

                    tree_tournees = cKDTree(coords_tournees)

                    for idx in df.index[valid_gps]:

                        tournee_actuelle = df.at[idx, "code_tournee"]

                        if tournee_actuelle not in refs_valides.index:
                            continue

                        x = df.at[idx, "x_client_original"]
                        y = df.at[idx, "y_client_original"]

                        # -------------------------------------------------------------
                        # Recherche des 2 tourn�es les plus proches
                        # -------------------------------------------------------------

                        distances, positions = tree_tournees.query(
                            [x, y],
                            k=min(2, len(refs_valides))
                        )

                        distances = np.atleast_1d(distances)
                        positions = np.atleast_1d(positions)

                        meilleure_autre_tournee = None
                        meilleure_distance = np.inf

                        for dist_deg, pos in zip(distances, positions):

                            candidate = refs_valides.index[pos]

                            if candidate == tournee_actuelle:
                                continue

                            # Conversion degr�s -> km
                            dist_km = float(dist_deg) * 111.0

                            if dist_km < meilleure_distance:
                                meilleure_distance = dist_km
                                meilleure_autre_tournee = candidate

                        if meilleure_autre_tournee is not None:

                            df.at[
                                idx,
                                "gps_tournee_proche"
                            ] = meilleure_autre_tournee

                            df.at[
                                idx,
                                "gps_distance_tournee_proche_km"
                            ] = meilleure_distance

                # -------------------------------------------------------------------------
                # D�tection anomalies
                # -------------------------------------------------------------------------

                seuil = config["distance_max_tournee_km"]

                anomalie_tournee = (
                    df["gps_distance_tournee_km"] > seuil
                )

                anomalie_changement_tournee = (
                    df["gps_distance_tournee_proche_km"]
                    <
                    df["gps_distance_tournee_km"] * 0.50
                ) & (
                    df["gps_distance_tournee_proche_km"]
                    <
                    config["distance_changement_tournee_km"]
                )

                df.loc[
                    anomalie_tournee,
                    "gps_anomalie"
                ] = True

                df.loc[
                    anomalie_tournee,
                    "gps_anomalie_type"
                ] = "HORS_TOURNEE"

                # Cas plus fort :
                # le client est beaucoup plus proche d'une autre tourn�e.
                df.loc[
                    anomalie_changement_tournee,
                    "gps_anomalie"
                ] = True

                df.loc[
                    anomalie_changement_tournee,
                    "gps_anomalie_type"
                ] = "TOURNEE_INCOHERENTE"

                logger.info(
                    f"Clients suspects par rapport � leur tourn�e : "
                    f"{int(anomalie_tournee.sum()):,}"
                )

            else:

                logger.warning(
                    "Colonne code_tournee absente : "
                    "contr�le des tourn�es d�sactiv�."
                )


            # =============================================================================
            # 2. CONTROLE HORS COMMUNE
            # =============================================================================
            #
            # Ce contr�le n�cessite un shapefile GeoJSON / SHP des communes.
            #
            # Exemple :
            #
            # GPS_CONFIG["fichier_communes"] = r"C:\SIG\communes.shp"
            #
            # =============================================================================

            fichier_communes = config.get("fichier_communes")

            if fichier_communes is not None:

                try:

                    import geopandas as gpd
                    from shapely.geometry import Point

                    if not os.path.exists(fichier_communes):

                        logger.warning(
                            f"Fichier communes introuvable : "
                            f"{fichier_communes}"
                        )

                    else:

                        logger.info(
                            "Chargement du r�f�rentiel g�ographique des communes..."
                        )

                        communes = gpd.read_file(
                            fichier_communes
                        )

                        colonne_commune = config.get(
                            "colonne_commune",
                            "NOM"
                        )

                        if colonne_commune not in communes.columns:

                            raise ValueError(
                                f"Colonne commune '{colonne_commune}' absente "
                                f"du fichier SIG."
                            )

                        # -------------------------------------------------------------
                        # CRS
                        # -------------------------------------------------------------

                        if communes.crs is None:

                            raise ValueError(
                                "Le fichier des communes ne poss�de pas de CRS."
                            )

                        # GPS = WGS84
                        communes = communes.to_crs(
                            "EPSG:4326"
                        )

                        # -------------------------------------------------------------
                        # Sous-ensemble clients valides
                        # -------------------------------------------------------------

                        gps_df = df.loc[
                            valid_gps,
                            [
                                "x_client_original",
                                "y_client_original"
                            ]
                        ].copy()

                        geometry = gpd.points_from_xy(
                            gps_df["x_client_original"],
                            gps_df["y_client_original"]
                        )

                        clients_geo = gpd.GeoDataFrame(
                            gps_df,
                            geometry=geometry,
                            crs="EPSG:4326"
                        )

                        # -------------------------------------------------------------
                        # Jointure spatiale
                        # -------------------------------------------------------------

                        clients_commune = gpd.sjoin(
                            clients_geo,
                            communes[
                                [colonne_commune, "geometry"]
                            ],
                            how="left",
                            predicate="within"
                        )

                        # -------------------------------------------------------------
                        # Attribution commune
                        # -------------------------------------------------------------

                        for idx, commune in clients_commune[
                            colonne_commune
                        ].items():

                            if pd.notna(commune):

                                df.at[
                                    idx,
                                    "gps_commune"
                                ] = commune

                            else:

                                df.at[
                                    idx,
                                    "gps_hors_commune"
                                ] = True

                                df.at[
                                    idx,
                                    "gps_anomalie"
                                ] = True

                                # Ne pas �craser une anomalie tourn�e
                                if df.at[
                                    idx,
                                    "gps_anomalie_type"
                                ] == "OK":

                                    df.at[
                                        idx,
                                        "gps_anomalie_type"
                                    ] = "HORS_COMMUNE"

                        nb_hors_commune = int(
                            df["gps_hors_commune"].sum()
                        )

                        logger.info(
                            f"Clients hors commune d�tect�s : "
                            f"{nb_hors_commune:,}"
                        )

                except ImportError:

                    logger.warning(
                        "GeoPandas/Shapely non install�. "
                        "Contr�le des communes ignor�."
                    )

                except Exception as e:

                    logger.exception(
                        f"Erreur lors du contr�le des communes : {e}"
                    )

            else:

                logger.info(
                    "Aucun fichier de communes fourni : "
                    "contr�le hors commune d�sactiv�."
                )


            # =============================================================================
            # 3. PROPOSITION DE CORRECTION
            # =============================================================================
            #
            # Pour les anomalies :
            #   on cherche des clients fiables de la m�me tourn�e.
            #
            # On utilise les K voisins les plus proches.
            #
            # IMPORTANT :
            #   on ne modifie PAS encore x_client / y_client.
            #
            # =============================================================================

            anomalies_idx = df.index[
                df["gps_anomalie"] &
                valid_gps
            ]

            logger.info(
                f"Recherche de positions correctives pour "
                f"{len(anomalies_idx):,} clients..."
            )

            if "code_tournee" in df.columns:

                for idx in anomalies_idx:

                    tournee = df.at[
                        idx,
                        "code_tournee"
                    ]

                    # Clients de r�f�rence de la m�me tourn�e
                    ref_mask = (
                        valid_gps
                        &
                        (df["code_tournee"] == tournee)
                        &
                        (~df["gps_anomalie"])
                    )

                    ref_idx = df.index[ref_mask]

                    if len(ref_idx) < config["min_clients_tournee"]:

                        continue

                    ref_coords = np.column_stack([
                        df.loc[
                            ref_idx,
                            "x_client_original"
                        ].values,

                        df.loc[
                            ref_idx,
                            "y_client_original"
                        ].values
                    ])

                    client_coord = np.array([
                        df.at[idx, "x_client_original"],
                        df.at[idx, "y_client_original"]
                    ])

                    tree = cKDTree(
                        ref_coords
                    )

                    k = min(
                        config["k_voisins_correction"],
                        len(ref_idx)
                    )

                    distances, positions = tree.query(
                        client_coord,
                        k=k
                    )

                    distances = np.atleast_1d(distances)
                    positions = np.atleast_1d(positions)

                    # -----------------------------------------------------------------
                    # Conversion en km
                    # -----------------------------------------------------------------

                    distances_km = distances * 111.0

                    # -----------------------------------------------------------------
                    # Poids inverse de la distance
                    # -----------------------------------------------------------------

                    weights = 1.0 / (
                        distances_km + 0.05
                    )

                    weights = weights / weights.sum()

                    voisins = ref_coords[positions]

                    # -----------------------------------------------------------------
                    # Position propos�e = moyenne pond�r�e des voisins
                    # -----------------------------------------------------------------

                    x_corrige = np.sum(
                        voisins[:, 0] * weights
                    )

                    y_corrige = np.sum(
                        voisins[:, 1] * weights
                    )

                    # -----------------------------------------------------------------
                    # Score de confiance
                    # -----------------------------------------------------------------

                    distance_moyenne = float(
                        np.average(
                            distances_km,
                            weights=weights
                        )
                    )

                    if distance_moyenne <= 0.5:

                        confiance = 0.95

                    elif distance_moyenne <= 1.0:

                        confiance = 0.90

                    elif distance_moyenne <= 2.0:

                        confiance = 0.80

                    elif distance_moyenne <= 5.0:

                        confiance = 0.65

                    else:

                        confiance = 0.40

                    df.at[
                        idx,
                        "x_client_corrige"
                    ] = x_corrige

                    df.at[
                        idx,
                        "y_client_corrige"
                    ] = y_corrige

                    df.at[
                        idx,
                        "gps_correction_proposee"
                    ] = True

                    df.at[
                        idx,
                        "gps_confiance_correction"
                    ] = confiance

                    df.at[
                        idx,
                        "gps_source_correction"
                    ] = (
                        "Voisins fiables m�me tourn�e"
                    )


            # =============================================================================
            # 4. APPLICATION OPTIONNELLE
            # =============================================================================

            corrections_fortes = (
                df["gps_correction_proposee"]
                &
                (
                    df["gps_confiance_correction"]
                    >=
                    config["seuil_confiance_correction"]
                )
            )

            if config["APPLIQUER_CORRECTIONS"]:

                df.loc[
                    corrections_fortes,
                    "x_client"
                ] = df.loc[
                    corrections_fortes,
                    "x_client_corrige"
                ]

                df.loc[
                    corrections_fortes,
                    "y_client"
                ] = df.loc[
                    corrections_fortes,
                    "y_client_corrige"
                ]

                logger.warning(
                    f"{int(corrections_fortes.sum()):,} "
                    f"positions GPS corrig�es automatiquement."
                )

            else:

                logger.info(
                    f"{int(corrections_fortes.sum()):,} "
                    f"corrections fortes propos�es."
                )

            # =============================================================================
            # 5. EXPORT DES ANOMALIES
            # =============================================================================

            colonnes_anomalies = [
                c for c in [
                    "code_client",
                    "code_tournee",

                    "x_client_original",
                    "y_client_original",

                    "x_client_corrige",
                    "y_client_corrige",

                    "gps_anomalie",
                    "gps_anomalie_type",

                    "gps_distance_tournee_km",

                    "gps_tournee_proche",
                    "gps_distance_tournee_proche_km",

                    "gps_commune",
                    "gps_hors_commune",

                    "gps_correction_proposee",
                    "gps_confiance_correction",
                    "gps_source_correction"
                ]
                if c in df.columns
            ]

            anomalies = (
                df.loc[
                    df["gps_anomalie"],
                    colonnes_anomalies
                ]
                .sort_values(
                    [
                        "gps_confiance_correction",
                        "gps_distance_tournee_km"
                    ],
                    ascending=[False, False]
                )
            )

            fichier_anomalies = os.path.join(
                CONFIG.output_path,
                "anomalies_positionnement_clients.csv"
            )

            anomalies.to_csv(
                fichier_anomalies,
                index=False,
                encoding="utf-8-sig"
            )

            logger.info(
                f"Rapport anomalies GPS : {fichier_anomalies}"
            )

            # =============================================================================
            # 6. STATISTIQUES FINALES
            # =============================================================================

            logger.info("-" * 70)
            logger.info("RESULTAT CONTROLE GPS")
            logger.info("-" * 70)

            logger.info(
                f"Total clients                 : {len(df):,}"
            )

            logger.info(
                f"Anomalies GPS                 : "
                f"{int(df['gps_anomalie'].sum()):,}"
            )

            logger.info(
                f"Hors commune                  : "
                f"{int(df['gps_hors_commune'].sum()):,}"
            )

            logger.info(
                f"Corrections propos�es         : "
                f"{int(df['gps_correction_proposee'].sum()):,}"
            )

            logger.info(
                f"Corrections haute confiance  : "
                f"{int(corrections_fortes.sum()):,}"
            )

            if len(anomalies) > 0:

                logger.info(
                    "\nR�partition des anomalies :\n"
                    +
                    anomalies[
                        "gps_anomalie_type"
                    ]
                    .value_counts()
                    .to_string()
                )

            return df, anomalies



        df, anom_corr = controler_positions_clients(df)


        df.rename(columns={
            'x_client': 'x_client_original',
            'y_client': 'y_client_original',
            'x_client_corrige': 'x_client',
            'y_client_corrige': 'y_client'
        }, inplace=True)


        #++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++

        # Variables metier incoherentes
        # [FIX] Les .clip() d'origine supposaient ces colonnes toujours
        # presentes (pas de garde `if col in df.columns`) -> KeyError si la
        # table source ne contenait pas exactement ces champs. On garde
        # desormais chaque operation.

        # Convertir l'annee d'installation en age reel (en annees)
        annee_actuelle = df['DATLCTDRN'].max().year
        df['age_compteur'] = annee_actuelle - df['age_compteur']

        clip_rules = {
            'diametre_compteur': (10, 500),
            'age_compteur': (0, 50),
            'temps_service': (0, 120),
            'taux_accessibilite': (0, 1),
            'taux_acces_zone': (0, 1),
            'fiabilite_agent': (0, 1),
        }
        for col, (lo, hi) in clip_rules.items():
            if col in df.columns:
                df[col] = pd.to_numeric(df[col], errors='coerce').clip(lower=lo, upper=hi)

        # [NEW] Nettoyage des variables financieres utilisees par les
        # PARTIES 6 et 7 (encaissement / recouvrement) : valeurs negatives
        # aberrantes (saisie/anomalie comptable) ramenees a 0, taux borne.
        for col in ['creance_totale', 'montant_encaisse', 'ca_2025']:
            if col in df.columns:
                df[col] = pd.to_numeric(df[col], errors='coerce').fillna(0).clip(lower=0)
        if 'taux_encaissement' in df.columns:
            df['taux_encaissement'] = df['taux_encaissement'].clip(0, 2)

        logger.info(f"Nettoyage : {n_before} -> {len(df)} clients")
        return df


class FeatureEngineer:
    """Feature engineering: variables spatiales, temporelles, comportementales,
    de risque et de recouvrement."""

    def __init__(self, config: PipelineConfig):
        self.config = config
        self.feature_names = []

    def engineer_features(self, df: pd.DataFrame) -> pd.DataFrame:
        """Cree l'ensemble des features du pipeline."""
        logger.info("=" * 60)
        logger.info("FEATURE ENGINEERING")
        logger.info("=" * 60)

        df = df.copy()

        df = self._spatial_features(df)
        df = self._temporal_features(df)
        df = self._comportemental_features(df)
        df = self._risque_features(df)
        df = self._optimisation_tournee_features(df)
        df = self._recouvrement_features(df)  # [NEW] point 2/3 du theme

        self.feature_names = [c for c in df.columns
                            if c not in ['code_client', 'id_controle', 'code_tournee',
                                        'id_agent', 'type_usage', 'type_facturation',
                                        'niveau_creance', 'date_plan_action']]

        logger.info(f"Features creees : {len(self.feature_names)}")
        return df

    def _spatial_features(self, df: pd.DataFrame) -> pd.DataFrame:
        """Features geospatiales (distance depot, densite, zones)."""

        # Distance client-depot
        # [FIX] robustesse : l'apply row-wise avec geodesic() sur un grand
        # dataframe est tres couteux et plante si x/y contiennent encore des
        # NaN residuels (geodesic leve un ValueError sur NaN). On calcule
        # d'abord un masque de validite, puis on applique geodesic
        # uniquement sur les lignes valides ; le reste recoit l'approximation
        # euclidienne (repli), garantissant qu'aucune ligne ne reste NaN.
        coord_cols = ['x_client', 'y_client', 'x_depot', 'y_depot']
        has_coords = all(c in df.columns for c in coord_cols)

        if has_coords:
            valid = df[coord_cols].notna().all(axis=1)
            df['distance_depot_km'] = np.sqrt(
                (df['x_client'] - df['x_depot']) ** 2 +
                (df['y_client'] - df['y_depot']) ** 2
            ) * 111  # approximation euclidienne par defaut (1 deg ~ 111 km)

            if GEOPY_AVAILABLE and valid.any():
                idx_valid = df.index[valid]
                df.loc[idx_valid, 'distance_depot_km'] = df.loc[idx_valid].apply(
                    lambda r: geodesic((r['y_client'], r['x_client']),
                                        (r['y_depot'], r['x_depot'])).kilometers,
                    axis=1
                )
        else:
            df['distance_depot_km'] = np.nan

        # Densite de compteurs dans un rayon de ~500m (approximation)
        if {'x_client', 'y_client'}.issubset(df.columns):
            coords_df = df[['x_client', 'y_client']].dropna()
            if len(coords_df) > 10:
                tree = cKDTree(coords_df.values)
                counts = tree.query_ball_point(coords_df.values, r=0.0045)
                voisins = pd.Series([len(c) - 1 for c in counts], index=coords_df.index)
                df['voisins_500m'] = voisins.reindex(df.index).fillna(0)
            else:
                df['voisins_500m'] = 0



            # Clustering spatial pour identifier les zones geographiques
            valid_xy = df[['x_client', 'y_client']].dropna()
            if len(valid_xy) > 10:
                coords_scaled = StandardScaler().fit_transform(valid_xy)
                kmeans = MiniBatchKMeans(n_clusters=self.config.n_clusters, random_state=42, batch_size=1000, n_init=3)
                zone_labels = pd.Series(kmeans.fit_predict(coords_scaled), index=valid_xy.index)
                df['zone_geo'] = zone_labels.reindex(df.index).fillna(-1).astype(int)
            else:
                df['zone_geo'] = -1

            # Centre de gravite de la zone + distance au centre (vectorise)
            zone_centers = df.groupby('zone_geo')[['x_client', 'y_client']].transform('mean')
            df['distance_zone_centre'] = np.sqrt(
                (df['x_client'] - zone_centers['x_client']) ** 2 +
                (df['y_client'] - zone_centers['y_client']) ** 2
            ) * 111

        logger.info("Features spatiales creees")
        return df

    def _temporal_features(self, df: pd.DataFrame) -> pd.DataFrame:
        """Features temporelles pour planification et saisonnalite."""

        df['conso_saisonniere'] = np.sin(2 * np.pi * pd.Timestamp.now().dayofyear / 365.25)

        if 'age_compteur' in df.columns:
            df['risque_panne'] = 1 / (1 + np.exp(-(df['age_compteur'] - 20) / 5))

        cycle_map = {'M01': 30, 'T01': 90, 'B01': 60}
        if 'cycle_facturation' in df.columns:
            # [FIX] `cycle_facturation` est de dtype `category` (converti en
            # PARTIE 2). `Series.map()` applique sur une Categorical renvoie
            # un resultat qui reste de dtype `category`, ce qui fait
            # planter toute operation arithmetique ulterieure
            # ("TypeError: Object with dtype category cannot perform the
            # numpy op multiply", reproduit lors des tests de ce pipeline).
            # On force explicitement une sortie numerique.
            df['cycle_jours'] = df['cycle_facturation'].astype(str).map(cycle_map).fillna(90).astype(float)
        else:
            df['cycle_jours'] = 90.0

        if {'taux_accessibilite'}.issubset(df.columns):
            df['jours_prochaine_releve'] = df['cycle_jours'] * (1 - df['taux_accessibilite'])

        logger.info("Features temporelles creees")
        return df

    def _comportemental_features(self, df: pd.DataFrame) -> pd.DataFrame:
        """Features de comportement client (fiabilite, priorite, alerte)."""

        if 'ecart_consommation' in df.columns:
            q95 = df['ecart_consommation'].quantile(0.95)
            df['instabilite_conso_norm'] = df['ecart_consommation'] / (q95 if q95 else 1)

        if {'fiabilite_agent', 'taux_accessibilite', 'variabilite_temps'}.issubset(df.columns):
            df['fiabilite_releve'] = (
                df['fiabilite_agent'] * 0.4 +
                df['taux_accessibilite'] * 0.4 +
                (1 - df['variabilite_temps'].clip(0, 1)) * 0.2
            )

        if {'indice_attractivite', 'indice_criticite', 'taux_accessibilite'}.issubset(df.columns):
            df['score_priorite'] = (
                df['indice_attractivite'] * 0.3 +
                df['indice_criticite'] * 0.4 +
                (1 - df['taux_accessibilite']) * 0.3
            ) * 100

        conditions = []
        if 'a_plan_action' in df.columns:
            conditions.append(df['a_plan_action'] == 1)
        if 'niveau_creance' in df.columns:
            conditions.append(df['niveau_creance'].isin(['Elevee', 'Critique']))
        if 'ecart_consommation' in df.columns:
            conditions.append(df['ecart_consommation'] > self.config.ecart_conso_max)
        if conditions:
            combined = conditions[0]
            for c in conditions[1:]:
                combined = combined | c
            df['client_problematique'] = combined.astype(int)
        else:
            df['client_problematique'] = 0

        logger.info("Features comportementales creees")
        return df

    def _risque_features(self, df: pd.DataFrame) -> pd.DataFrame:
        """Features de risque financier et technique (base du score global)."""

        if {'creance_totale', 'ca_2025', 'taux_encaissement', 'delai_dernier_encaissement'}.issubset(df.columns):
            df['risque_non_paiement'] = (
                (df['creance_totale'] / df['ca_2025'].replace(0, np.nan)).fillna(0).clip(0, 10) *
                (1 - df['taux_encaissement']) *
                (df['delai_dernier_encaissement'] / 365).clip(0, 3)
            )
        else:
            df['risque_non_paiement'] = 0.0

        if {'indice_criticite', 'fiabilite_releve', 'ecart_consommation'}.issubset(df.columns):
            df['risque_erreur_facturation'] = (
                df['indice_criticite'] * 0.5 +
                (1 - df['fiabilite_releve']) * 0.3 +
                df['ecart_consommation'].clip(0, 5) / 5 * 0.2
            )
        else:
            df['risque_erreur_facturation'] = 0.0

        q95_risque = df['risque_non_paiement'].quantile(0.95)
        df['score_risque_global'] = (
            df['risque_non_paiement'] / (q95_risque if q95_risque else 1) * 25 +
            df['risque_erreur_facturation'] * 25 +
            df.get('indice_criticite', 0) * 25 +
            (1 - df.get('taux_accessibilite', 0)) * 25
        ).clip(0, 100)

        df['categorie_risque'] = pd.cut(
            df['score_risque_global'],
            bins=[0, 25, 50, 75, 100],
            labels=['Faible', 'Moyen', 'Eleve', 'Critique'],
            include_lowest=True
        )

        logger.info("Features de risque creees")
        return df

    def _optimisation_tournee_features(self, df: pd.DataFrame) -> pd.DataFrame:
        """Features specifiques a l'optimisation des tournees (TSP/VRP)."""

        if {'indice_attractivite', 'taux_accessibilite', 'risque_panne', 'score_risque_global'}.issubset(df.columns):
            df['urgence_releve'] = (
                df['indice_attractivite'] * 0.3 +
                (1 - df['taux_accessibilite']) * 0.3 +
                df['risque_panne'] * 0.2 +
                df['score_risque_global'] / 100 * 0.2
            )
        else:
            df['urgence_releve'] = df.get('score_risque_global', 0) / 100

        if {'temps_service', 'temps_trajet'}.issubset(df.columns):
            df['temps_total_estime'] = df['temps_service'] + df['temps_trajet']
        else:
            df['temps_total_estime'] = df.get('temps_service', 15)

        df['productivite_attendue'] = (
            df.get('taux_accessibilite', 0.5) *
            (1 / df['temps_total_estime'].replace(0, np.nan)).fillna(0) * 60
        )

        if 'taux_accessibilite' in df.columns:
            df['creneau_recommande'] = np.where(
                df['taux_accessibilite'] > 0.8, 'Matin',
                np.where(df['taux_accessibilite'] > 0.5, 'Flexible', 'Apres-midi')
            )

        if 'variabilite_temps' in df.columns:
            df['jour_recommande'] = np.where(
                df['variabilite_temps'] > df['variabilite_temps'].quantile(0.75),
                'Eviter weekend', 'Tous jours'
            )

        logger.info("Features d'optimisation de tournee creees")
        return df

    def _recouvrement_features(self, df: pd.DataFrame) -> pd.DataFrame:
        """[NEW] Features dediees au recouvrement (theme point 2 et 3) :
        indicateurs de comportement de paiement utilises a la fois par la
        prevision des encaissements (PARTIE 6) et par la segmentation
        strategique (PARTIE 7)."""

        # Anciennete de creance relative au CA (poids de la dette)
        if {'creance_totale', 'ca_2025'}.issubset(df.columns):
            df['poids_creance_ca'] = (
                df['creance_totale'] / df['ca_2025'].replace(0, np.nan)
            ).fillna(0).clip(0, 10)

        # Regularite de paiement : delai depuis le dernier encaissement,
        # normalise par le cycle de facturation du client
        if {'delai_dernier_encaissement', 'cycle_jours'}.issubset(df.columns):
            df['retard_relatif_paiement'] = (
                df['delai_dernier_encaissement'] / df['cycle_jours'].replace(0, np.nan)
            ).fillna(0).clip(0, 20)

        # Client "bon payeur" vs "a risque" (variable cible auxiliaire pour
        # l'evaluation des strategies de recouvrement, PARTIE 7)
        if 'taux_encaissement' in df.columns:
            df['bon_payeur'] = (df['taux_encaissement'] >= self.config.seuil_taux_encaissement_bas).astype(int)

        logger.info("Features de recouvrement creees")
        return df

In [8]:
# Nettoyage des données
cleaner = DataCleaner(CONFIG)
df_cleaned = cleaner.clean(df)

2026-09-24 10:02:51,307 [INFO] ============================================================
2026-09-24 10:02:51,307 [INFO] NETTOYAGE DES DONNEES
2026-09-24 10:02:51,307 [INFO] ============================================================
2026-09-24 10:02:51,535 [INFO] ======================================================================
2026-09-24 10:02:51,535 [INFO] CONTROLE DES POSITIONS GPS CLIENTS
2026-09-24 10:02:51,537 [INFO] ======================================================================
2026-09-24 10:02:51,543 [INFO] Clients avec GPS valide : 84,590/84,590
2026-09-24 10:02:51,543 [INFO] Analyse de coh�rence g�ographique par tourn�e...
2026-09-24 10:03:28,162 [INFO] Clients suspects par rapport � leur tourn�e : 238
2026-09-24 10:03:28,167 [INFO] Aucun fichier de communes fourni : contr�le hors commune d�sactiv�.
2026-09-24 10:03:28,169 [INFO] Recherche de positions correctives pour 7,958 clients...
2026-09-24 10:03:36,663 [INFO] 7,391 corrections fortes propos�es.
2026-09

In [9]:
df_cleaned.isnull().sum().sort_values(ascending=False)

gps_commune                             84590
date_plan_action                        20990
date_encaissement                       17436
code_observation                          401
diametre_compteur                         399
age_compteur                              399
DATLCTPRC                                 399
DATLCTDRN                                 399
jour RLV                                  399
gps_tournee_proche                         10
gps_distance_tournee_proche_km             10
gps_distance_tournee_km                    10
y_client_original                           0
delai_dernier_encaissement                  0
montant_encaisse                            0
ca_2025                                     0
taux_encaissement                           0
niveau_creance                              0
creance_totale                              0
x_client_original                           0
code_client                                 0
gps_anomalie                      

In [10]:
# Dropping rows with missing values in 'age_compteur', 'code_observation'

df_cleaned = df_cleaned.dropna(subset=['age_compteur', 'code_observation'])

In [11]:
df_cleaned.isnull().sum().sort_values(ascending=False)

gps_commune                             84189
date_plan_action                        20910
date_encaissement                       17084
gps_distance_tournee_proche_km              7
gps_tournee_proche                          7
gps_distance_tournee_km                     7
code_client                                 0
delai_dernier_encaissement                  0
action_Relancer le client via CATO          0
action_Télé-Relance                         0
a_plan_action                               0
creance_totale                              0
niveau_creance                              0
ca_2025                                     0
montant_encaisse                            0
y_client_original                           0
taux_encaissement                           0
x_client_original                           0
action_Relance SMS Exigibilité              0
gps_anomalie                                0
gps_anomalie_type                           0
gps_hors_commune                  

In [12]:
df_cleaned['age_compteur'].unique()

array([ 5.,  2., 12.,  8.,  7., 11.,  6.,  3., 13.,  1., 10.,  9.,  4.,
       16., 14., 15.,  0., 22., 17., 20., 19., 18., 21., 26.])

In [13]:
# Ingénierie des caractéristiques

feature_engineer = FeatureEngineer(CONFIG)
df_features = feature_engineer.engineer_features(df_cleaned)

2026-09-24 10:03:36,963 [INFO] ============================================================
2026-09-24 10:03:36,963 [INFO] FEATURE ENGINEERING
2026-09-24 10:03:36,963 [INFO] ============================================================
2026-09-24 10:04:57,832 [INFO] Features spatiales creees
2026-09-24 10:05:03,761 [INFO] Features temporelles creees
2026-09-24 10:05:03,767 [INFO] Features comportementales creees
2026-09-24 10:05:03,821 [INFO] Features de risque creees
2026-09-24 10:05:03,831 [INFO] Features d'optimisation de tournee creees
2026-09-24 10:05:03,843 [INFO] Features de recouvrement creees
2026-09-24 10:05:03,843 [INFO] Features creees : 63


In [14]:
df_features.columns

Index(['code_client', 'id_agent', 'adresse_pntcpg', 'x_client_original', 'y_client_original', 'x_depot', 'y_depot', 'code_tournee', 'cycle_facturation', 'id_controle', 'type_usage',
       'type_facturation', 'diametre_compteur', 'DATLCTDRN', 'DATLCTPRC', 'jour RLV', 'age_compteur', 'code_observation', 'volume_facture', 'date_plan_action', 'action_Coupures',
       'action_Envoi E-mail', 'action_Juridique', 'action_Mise en Demeure', 'action_Relance Avocat', 'action_Relance Gestion Personnalisée', 'action_Relance Porte à porte',
       'action_Relance SMS Exigibilité', 'action_Relance Systématique', 'action_Relancer le client via CATO', 'action_Télé-Relance', 'a_plan_action', 'creance_totale', 'niveau_creance', 'ca_2025',
       'montant_encaisse', 'date_encaissement', 'delai_dernier_encaissement', 'taux_encaissement', 'x_client_original', 'y_client_original', 'gps_anomalie', 'gps_anomalie_type',
       'gps_distance_tournee_km', 'gps_tournee_proche', 'gps_distance_tournee_proche_km', '

In [15]:
df_features['score_risque_global']

0         30.745813
2         24.977095
5         25.000000
8         25.004073
10        25.000049
            ...    
250026    25.000000
250027    25.000000
250028    25.000000
250156    26.279169
250646    25.000000
Name: score_risque_global, Length: 84189, dtype: float64

## **PARTIE 4 : Analyse Exploratoire (EDA)**

In [16]:
# ==============================================================================
# PARTIE 4 : ANALYSE EXPLORATOIRE (EDA)
# ==============================================================================

class EDAAnalyzer:
    """EDA sur les donnees clients / compteurs / recouvrement."""

    def __init__(self, config: PipelineConfig):
        self.config = config

    def analyze(self, df: pd.DataFrame):
        """Pipeline EDA complet."""
        logger.info("=" * 60)
        logger.info("ANALYSE EXPLORATOIRE")
        logger.info("=" * 60)

        self._distribution_metiers(df)
        self._analyse_spatiale(df)
        self._analyse_performance_tournees(df)
        self._analyse_risques(df)
        self._correlations(df)

    def _distribution_metiers(self, df: pd.DataFrame):
        """Distributions des variables metier cles."""
        fig, axes = plt.subplots(2, 2, figsize=(16, 10))

        # if 'taux_accessibilite' in df.columns:
        #     df['taux_accessibilite'].hist(bins=50, ax=axes[0, 0], color='steelblue', edgecolor='black')
        #     axes[0, 0].axvline(df['taux_accessibilite'].mean(), color='red', linestyle='--',
        #                         label=f'Moy: {df["taux_accessibilite"].mean():.2f}')
        #     axes[0, 0].set_title("Taux d'Accessibilite")
        #     axes[0, 0].legend()

        # if 'temps_service' in df.columns:
        #     df['temps_service'].hist(bins=50, ax=axes[0, 1], color='forestgreen', edgecolor='black')
        #     axes[0, 1].set_title('Temps de Service (min)')

        if 'volume_facture' in df.columns:
            df['volume_facture'].hist(bins=50, ax=axes[0, 0], color='coral', edgecolor='black')
            axes[0, 0].set_title('Volume Facture (m3)')

        if 'diametre_compteur' in df.columns:
            df['diametre_compteur'].hist(bins=30, ax=axes[0, 1], color='mediumpurple', edgecolor='black')
            axes[0, 1].set_title('Diametre Compteur (mm)')

        if 'age_compteur' in df.columns:
            df['age_compteur'].hist(bins=30, ax=axes[1, 0], color='gold', edgecolor='black')
            axes[1, 0].set_title('Age Compteur (annees)')

        if 'score_risque_global' in df.columns:
            df['score_risque_global'].hist(bins=50, ax=axes[1, 1], color='crimson', edgecolor='black')
            axes[1, 1].axvline(75, color='black', linestyle='--', label='Seuil critique')
            axes[1, 1].set_title('Score de Risque Global')
            axes[1, 1].legend()

        plt.tight_layout()
        plt.savefig(os.path.join(self.config.output_path, 'eda_distributions.png'), dpi=150)
        plt.close()
        logger.info("Distributions metier generees")

    def _analyse_spatiale(self, df: pd.DataFrame):
        """Cartographie des taux d'accessibilite et des risques."""
        if not FOLIUM_AVAILABLE or not {'x_client', 'y_client'}.issubset(df.columns):
            return

        map_center = [df['x_client'].mean(), df['y_client'].mean()]
        m = folium.Map(location=map_center, zoom_start=12, tiles='CartoDB positron')

        sample = df.sample(min(3000, len(df)), random_state=RANDOM_STATE)
        for _, row in sample.iterrows():
            access = row.get('taux_accessibilite', np.nan)
            color = 'green' if access > 0.8 else 'orange' if access > 0.5 else 'red'
            CircleMarker(
                location=[row['x_client'], row['y_client']],
                radius=4, color=color, fill=True, fill_opacity=0.6,
                popup=f"Client: {row['code_client']}<br>Access: {access:.1%}<br>"
                      f"Risque: {row.get('score_risque_global', np.nan):.1f}"
            ).add_to(m)

        if 'taux_accessibilite' in df.columns:
            low_access = df[df['taux_accessibilite'] < 0.5]
            if len(low_access) > 0:
                HeatMap(low_access[['x_client', 'y_client']].values, radius=20, blur=15).add_to(m)

        m.save(os.path.join(self.config.output_path, 'carte_accessibilite.html'))

        if 'categorie_risque' in df.columns:
            m2 = folium.Map(location=map_center, zoom_start=12, tiles='CartoDB dark_matter')
            risk_colors = {'Faible': 'green', 'Moyen': 'yellow', 'Eleve': 'orange', 'Critique': 'red'}
            for _, row in sample.iterrows():
                color = risk_colors.get(str(row.get('categorie_risque')), 'grey')
                CircleMarker(
                    location=[row['x_client'], row['y_client']],
                    radius=3 + row.get('score_risque_global', 0) / 20,
                    color=color, fill=True, fill_opacity=0.7
                ).add_to(m2)
            m2.save(os.path.join(self.config.output_path, 'carte_risques.html'))

        logger.info("Cartes spatiales generees")

    def _analyse_performance_tournees(self, df: pd.DataFrame):
        """Analyse de la performance des tournees par agent."""
        if not {'id_agent', 'taux_accessibilite', 'temps_service'}.issubset(df.columns):
            return

        fig, axes = plt.subplots(2, 2, figsize=(14, 10))

        perf_agent = df.groupby('id_agent').agg({
            'taux_accessibilite': 'mean',
            'temps_service': 'mean',
            'fiabilite_agent': 'first',
            'code_client': 'count'
        }).rename(columns={'code_client': 'nb_clients'})
        perf_agent = perf_agent[perf_agent['nb_clients'] >= 10]

        perf_agent['taux_accessibilite'].sort_values().plot(kind='barh', ax=axes[0, 0], color='steelblue')
        axes[0, 0].set_title("Taux d'Accessibilite Moyen par Agent")
        axes[0, 0].axvline(0.7, color='red', linestyle='--', label='Seuil critique')

        axes[0, 1].scatter(perf_agent['fiabilite_agent'], perf_agent['taux_accessibilite'],
                            s=perf_agent['nb_clients'] * 2, alpha=0.6, c='forestgreen')
        axes[0, 1].set_xlabel('Fiabilite Agent')
        axes[0, 1].set_ylabel('Taux Accessibilite')
        axes[0, 1].set_title('Fiabilite vs Accessibilite (taille = nb clients)')

        if 'productivite_attendue' in df.columns:
            axes[1, 0].scatter(df['temps_service'], df['productivite_attendue'],
                                c=df['taux_accessibilite'], cmap='RdYlGn', alpha=0.5)
            axes[1, 0].set_xlabel('Temps de Service (min)')
            axes[1, 0].set_ylabel('Productivite Attendue')
            axes[1, 0].set_title('Productivite vs Temps de Service')

        if 'type_usage' in df.columns:
            usage_stats = df.groupby('type_usage')['taux_accessibilite'].mean().sort_values()
            usage_stats.plot(kind='bar', ax=axes[1, 1], color='coral')
            axes[1, 1].set_title("Accessibilite par Type d'Usage")
            axes[1, 1].tick_params(axis='x', rotation=45)

        plt.tight_layout()
        plt.savefig(os.path.join(self.config.output_path, 'eda_performance_tournees.png'), dpi=150)
        plt.close()
        logger.info("Analyse performance tournees generee")

    def _analyse_risques(self, df: pd.DataFrame):
        """Analyse des risques clients / recouvrement."""
        if not {'categorie_risque', 'score_risque_global'}.issubset(df.columns):
            return

        fig, axes = plt.subplots(2, 2, figsize=(14, 10))

        risk_counts = df['categorie_risque'].value_counts()
        colors = ['#2ecc71', '#f1c40f', '#e67e22', '#e74c3c']
        axes[0, 0].pie(risk_counts, labels=risk_counts.index, autopct='%1.1f%%',
                        colors=colors[:len(risk_counts)], startangle=90)
        axes[0, 0].set_title('Repartition des Categories de Risque')

        if {'creance_totale', 'taux_encaissement'}.issubset(df.columns):
            axes[0, 1].scatter(df['creance_totale'], df['score_risque_global'],
                                c=df['taux_encaissement'], cmap='RdYlGn', alpha=0.5)
            axes[0, 1].set_xlabel('Creance Totale')
            axes[0, 1].set_ylabel('Score Risque Global')
            axes[0, 1].set_title('Risque vs Creance (couleur = taux encaissement)')
            axes[0, 1].set_xlim(0, df['creance_totale'].quantile(0.95))

        if 'a_plan_action' in df.columns:
            action_risk = df.groupby('a_plan_action')['score_risque_global'].mean()
            axes[1, 0].bar(['Sans action', 'Avec action'], action_risk, color=['steelblue', 'crimson'])
            axes[1, 0].set_title("Score Risque Moyen : Avec/Sans Plan d'Action")

        if {'age_compteur', 'risque_panne'}.issubset(df.columns):
            age_risk = df.groupby(pd.cut(df['age_compteur'], bins=10))['risque_panne'].mean()
            age_risk.plot(ax=axes[1, 1], color='purple', marker='o')
            axes[1, 1].set_title('Risque de Panne par Age de Compteur')
            axes[1, 1].set_xlabel('Age Compteur (annees)')

        plt.tight_layout()
        plt.savefig(os.path.join(self.config.output_path, 'eda_analyse_risques.png'), dpi=150)
        plt.close()
        logger.info("Analyse des risques generee")

    def _correlations(self, df: pd.DataFrame):
        """Matrice de correlation des variables cles."""
        cols_corr = ['taux_accessibilite', 'temps_service', 'temps_trajet',
                     'indice_attractivite', 'ecart_consommation', 'indice_criticite',
                     'fiabilite_agent', 'taux_acces_zone', 'variabilite_temps',
                     'indice_proximite', 'score_risque_global', 'urgence_releve',
                     'productivite_attendue', 'distance_depot_km',
                     'creance_totale', 'taux_encaissement']  # [NEW] variables recouvrement

        cols_corr = [c for c in cols_corr if c in df.columns]
        corr = df[cols_corr].corr()

        fig, ax = plt.subplots(figsize=(12, 10))
        mask = np.triu(np.ones_like(corr, dtype=bool))
        sns.heatmap(corr, mask=mask, annot=True, fmt='.2f', cmap='coolwarm',
                    center=0, ax=ax, square=True, linewidths=0.5, vmin=-1, vmax=1)
        ax.set_title('Matrice de Correlation - Variables Cles')
        plt.tight_layout()
        plt.savefig(os.path.join(self.config.output_path, 'eda_correlations.png'), dpi=150)
        plt.close()
        logger.info("Matrice de correlation generee")

In [17]:
df_features.columns

Index(['code_client', 'id_agent', 'adresse_pntcpg', 'x_client_original', 'y_client_original', 'x_depot', 'y_depot', 'code_tournee', 'cycle_facturation', 'id_controle', 'type_usage',
       'type_facturation', 'diametre_compteur', 'DATLCTDRN', 'DATLCTPRC', 'jour RLV', 'age_compteur', 'code_observation', 'volume_facture', 'date_plan_action', 'action_Coupures',
       'action_Envoi E-mail', 'action_Juridique', 'action_Mise en Demeure', 'action_Relance Avocat', 'action_Relance Gestion Personnalisée', 'action_Relance Porte à porte',
       'action_Relance SMS Exigibilité', 'action_Relance Systématique', 'action_Relancer le client via CATO', 'action_Télé-Relance', 'a_plan_action', 'creance_totale', 'niveau_creance', 'ca_2025',
       'montant_encaisse', 'date_encaissement', 'delai_dernier_encaissement', 'taux_encaissement', 'x_client_original', 'y_client_original', 'gps_anomalie', 'gps_anomalie_type',
       'gps_distance_tournee_km', 'gps_tournee_proche', 'gps_distance_tournee_proche_km', '

In [18]:
eda = EDAAnalyzer(CONFIG)
eda.analyze(df_features)

2026-09-24 10:05:04,140 [INFO] ============================================================
2026-09-24 10:05:04,140 [INFO] ANALYSE EXPLORATOIRE
2026-09-24 10:05:04,151 [INFO] ============================================================
2026-09-24 10:05:05,415 [INFO] Distributions metier generees
2026-09-24 10:05:11,431 [INFO] Cartes spatiales generees
2026-09-24 10:05:14,759 [INFO] Analyse des risques generee
2026-09-24 10:05:15,158 [INFO] Matrice de correlation generee


## **PARTIE 5 : DÉTECTION ET TYPOLOGIE DES ANOMALIES**

Alignement avec le theme de stage (point 1) :
- Detection non supervisee (Isolation Forest, LOF) **+** detection/typologie
  supervisee par **Random Forest** **+** detection par **reseau de
  neurones** (autoencodeur), le tout combine en un score d'ensemble.
- Classification du type d'anomalie (fraude, erreur de releve, panne,
  compteur degrade) a partir de caracteristiques statistiques et
  comportementales.
- Integration spatio-temporelle (zone geographique, historique de
  consommation).
- Systeme d'alerte a score de risque pour prioriser les investigations
  terrain.
- Cartographie interactive des zones a risque + graphiques de series
  temporelles / incoherences spatiales.

In [19]:
# ==============================================================================
# PARTIE 5 : DETECTION D'ANOMALIES
# ==============================================================================

class AnomalyDetector:
    """Detection et typologie des anomalies (consommation, releve, paiement)
    par ensemble de methodes non supervisees (Isolation Forest, LOF) et
    supervisees (Random Forest, reseau de neurones / autoencodeur)."""

    def __init__(self, config: PipelineConfig):
        self.config = config
        self.models = {}
        self.scaler = None
        self.feature_cols = []

    def detect(self, df: pd.DataFrame) -> pd.DataFrame:
        """Pipeline complet de detection d'anomalies."""
        logger.info("=" * 60)
        logger.info("DETECTION D'ANOMALIES")
        logger.info("=" * 60)

        df = df.copy()

        # Features pour la detection : dimensions techniques (compteur,
        # releve) ET dimensions financieres/comportementales (recouvrement),
        # ces dernieres ayant ete ecartees dans la version d'origine alors
        # qu'elles permettent de detecter des anomalies de paiement
        # (montant encaisse incoherent avec le CA facture, etc.).
        feature_cols = [
            'volume_facture', 'ecart_consommation', 'temps_service',
            'taux_accessibilite', 'indice_criticite', 'fiabilite_agent',
            'variabilite_temps', 'distance_depot_km', 'age_compteur',
            'diametre_compteur', 'densite_compteurs_km',
            # [NEW] dimension financiere / recouvrement
            'creance_totale', 'taux_encaissement', 'delai_dernier_encaissement',
        ]
        feature_cols = [c for c in feature_cols if c in df.columns]
        self.feature_cols = feature_cols

        X = df[feature_cols].fillna(df[feature_cols].median())
        scaler = StandardScaler()
        X_scaled = scaler.fit_transform(X)
        self.scaler = scaler

        # 1. Isolation Forest (non supervise)
        df = self._isolation_forest(df, X_scaled)

        # 2. LOF (non supervise, densite locale)
        df = self._lof(df, X_scaled)

        # 3. [NEW] Autoencodeur / reseau de neurones (non supervise, erreur
        #    de reconstruction)
        df = self._neural_network_autoencoder(df, X_scaled)

        # 4. Score d'ensemble non supervise (IF + LOF + NN)
        df = self._ensemble_score(df)

        # 5. Typologie metier des anomalies (regles + caracteristiques
        #    spatio-temporelles)
        df = self._classify_anomalies_metier(df)

        # 6. [NEW] Random Forest supervise : reprend les pseudo-labels issus
        #    de l'etape 5 pour (a) classer le TYPE d'anomalie a partir des
        #    caracteristiques statistiques/comportementales et (b) fournir
        #    l'importance des variables (explicabilite demandee par le theme)
        df = self._random_forest_typologie(df, X)

        # 7. [NEW] Score d'alerte final + priorisation des investigations
        #    terrain (systeme d'alerte du theme point 1)
        df = self._score_alerte(df)

        # 8. [NEW] Visualisations spatio-temporelles + carte interactive
        self._visualiser_anomalies(df)

        return df

    # ------------------------------------------------------------------
    # Methodes non supervisees
    # ------------------------------------------------------------------
    def _isolation_forest(self, df: pd.DataFrame, X: np.ndarray) -> pd.DataFrame:
        model = IsolationForest(
            n_estimators=200, contamination=self.config.contamination,
            random_state=RANDOM_STATE, n_jobs=-1
        )
        model.fit(X)
        scores = model.decision_function(X)

        df['anomaly_score_if'] = (1 - (scores - scores.min()) / (scores.max() - scores.min() + 1e-8)) * 100
        df['anomaly_pred_if'] = (model.predict(X) == -1).astype(int)

        self.models['isolation_forest'] = model
        logger.info(f"Isolation Forest : {df['anomaly_pred_if'].sum()} anomalies")
        return df

    def _lof(self, df: pd.DataFrame, X: np.ndarray) -> pd.DataFrame:
        model = LocalOutlierFactor(
            n_neighbors=20, contamination=self.config.contamination, n_jobs=-1
        )
        predictions = model.fit_predict(X)
        scores = model.negative_outlier_factor_

        df['anomaly_score_lof'] = (1 - (scores - scores.min()) / (scores.max() - scores.min() + 1e-8)) * 100
        df['anomaly_pred_lof'] = (predictions == -1).astype(int)

        logger.info(f"LOF : {df['anomaly_pred_lof'].sum()} anomalies")
        return df

    def _neural_network_autoencoder(self, df: pd.DataFrame, X: np.ndarray) -> pd.DataFrame:
        """[NEW] Detection d'anomalies par reseau de neurones : un
        autoencodeur est entraine a reconstruire les observations
        "normales" ; les clients dont l'erreur de reconstruction est
        elevee sont des anomalies potentielles. Utilise Keras si
        TensorFlow est disponible, sinon repli automatique sur un
        MLPRegressor scikit-learn configure en autoencodeur (memes
        principes, sans dependance lourde)."""

        n_features = X.shape[1]
        encoding_dim = max(2, n_features // 2)

        if TF_AVAILABLE:
            logger.info("Entrainement de l'autoencodeur (Keras/TensorFlow)...")
            tf.random.set_seed(RANDOM_STATE)
            np.random.seed(RANDOM_STATE)
            
            inputs = Input(shape=(n_features,))
            encoded = Dense(max(4, n_features), activation='relu')(inputs)
            encoded = Dense(encoding_dim, activation='relu')(encoded)
            decoded = Dense(max(4, n_features), activation='relu')(encoded)
            decoded = Dense(n_features, activation='linear')(decoded)

            autoencoder = Model(inputs, decoded)
            autoencoder.compile(optimizer=Adam(learning_rate=1e-3), loss='mse')

            early_stop = EarlyStopping(monitor='loss', patience=8, restore_best_weights=True)
            autoencoder.fit(
                X, X, epochs=100, batch_size=64, shuffle=True,
                verbose=0, callbacks=[early_stop]
            )
            X_reconstruit = autoencoder.predict(X, verbose=0)
            self.models['autoencodeur'] = autoencoder
        else:
            logger.info("TensorFlow indisponible : autoencodeur repli sur MLPRegressor (sklearn)...")
            # [FIX] meme precaution que dans SeriesTemporellesPredictor :
            # `early_stopping=True` echoue si l'echantillon est trop petit.
            early_stop_ok = len(X) >= 50
            mlp = MLPRegressor(
                hidden_layer_sizes=(max(4, n_features), encoding_dim, max(4, n_features)),
                activation='relu', solver='adam', max_iter=500,
                random_state=RANDOM_STATE, early_stopping=early_stop_ok
            )
            mlp.fit(X, X)
            X_reconstruit = mlp.predict(X)
            self.models['autoencodeur'] = mlp

        erreur_reconstruction = np.mean((X - X_reconstruit) ** 2, axis=1)
        seuil = np.quantile(erreur_reconstruction, 1 - self.config.contamination)

        e_min, e_max = erreur_reconstruction.min(), erreur_reconstruction.max()
        df['anomaly_score_nn'] = (erreur_reconstruction - e_min) / (e_max - e_min + 1e-8) * 100
        df['anomaly_pred_nn'] = (erreur_reconstruction > seuil).astype(int)

        logger.info(f"Reseau de neurones (autoencodeur) : {df['anomaly_pred_nn'].sum()} anomalies "
                     f"(seuil erreur reconstruction = {seuil:.4f})")
        return df

    def _ensemble_score(self, df: pd.DataFrame) -> pd.DataFrame:
        score_cols = ['anomaly_score_if', 'anomaly_score_lof', 'anomaly_score_nn']
        score_cols = [c for c in score_cols if c in df.columns]
        df['anomaly_score_ensemble'] = df[score_cols].mean(axis=1)

        pred_cols = ['anomaly_pred_if', 'anomaly_pred_lof', 'anomaly_pred_nn']
        pred_cols = [c for c in pred_cols if c in df.columns]
        # Vote majoritaire : anomalie si detectee par au moins 2 methodes
        # sur 3 (plus robuste que le "au moins 1" d'origine, qui generait
        # beaucoup de faux positifs des qu'une seule methode etait bruitee).
        seuil_vote = max(1, int(np.ceil(len(pred_cols) / 2)))
        df['anomaly_pred_ensemble'] = (df[pred_cols].sum(axis=1) >= seuil_vote).astype(int)

        logger.info(f"Ensemble (IF + LOF + NN, vote >= {seuil_vote}/{len(pred_cols)}) : "
                     f"{df['anomaly_pred_ensemble'].sum()} anomalies detectees")
        return df

    def _classify_anomalies_metier(self, df: pd.DataFrame) -> pd.DataFrame:
        """Classifie les anomalies selon leur type metier, en combinant
        criteres techniques ET spatio-temporels (ecart au comportement de
        la zone geographique - theme point 1 : 'analyse spatiale')."""

        # Ecart de consommation du client par rapport a sa zone geographique
        # (incoherence spatiale) - complete l'analyse purement individuelle.
        if {'zone_geo', 'ecart_consommation'}.issubset(df.columns):
            zone_mean = df.groupby('zone_geo')['ecart_consommation'].transform('mean')
            zone_std = df.groupby('zone_geo')['ecart_consommation'].transform('std').replace(0, np.nan)
            df['ecart_zone_zscore'] = ((df['ecart_consommation'] - zone_mean) / zone_std).fillna(0)
        else:
            df['ecart_zone_zscore'] = 0.0

        df['anomalie_panne'] = (
            (df.get('volume_facture', 1) == 0) &
            (df.get('age_compteur', 0) > 15) &
            (df['anomaly_pred_ensemble'] == 1)
        ).astype(int)

        df['anomalie_fraude'] = (
            (df.get('ecart_consommation', 0) < -2) &
            (df.get('volume_facture', 0) > 0) &
            (df['anomaly_pred_ensemble'] == 1)
        ).astype(int)

        df['anomalie_erreur_releve'] = (
            (df.get('variabilite_temps', 0) > df.get('variabilite_temps', pd.Series([0])).quantile(0.9)) &
            (df.get('taux_accessibilite', 1) < 0.5) &
            (df['anomaly_pred_ensemble'] == 1)
        ).astype(int)

        df['anomalie_compteur_degrade'] = (
            (df.get('indice_criticite', 0) > 0.7) &
            (df.get('fiabilite_agent', 1) < 0.5) &
            (df['anomaly_pred_ensemble'] == 1)
        ).astype(int)

        # [NEW] Anomalie de paiement / encaissement : creance elevee mais
        # taux d'encaissement anormalement bas malgre relance recente
        # (theme : "optimisation des encaissements et recouvrement").
        if {'creance_totale', 'taux_encaissement', 'a_plan_action'}.issubset(df.columns):
            df['anomalie_encaissement'] = (
                (df['creance_totale'] > df['creance_totale'].quantile(0.9)) &
                (df['taux_encaissement'] < self.config.seuil_taux_encaissement_bas) &
                (df['a_plan_action'] == 1) &
                (df['anomaly_pred_ensemble'] == 1)
            ).astype(int)
        else:
            df['anomalie_encaissement'] = 0

        # [NEW] Incoherence spatiale : consommation tres eloignee de la
        # moyenne de sa zone geographique (theme : "incoherences spatiales")
        df['anomalie_incoherence_spatiale'] = (
            (df['ecart_zone_zscore'].abs() > 2.5) &
            (df['anomaly_pred_ensemble'] == 1)
        ).astype(int)

        type_cols = ['anomalie_panne', 'anomalie_fraude', 'anomalie_erreur_releve',
                     'anomalie_compteur_degrade', 'anomalie_encaissement',
                     'anomalie_incoherence_spatiale']
        for col in type_cols:
            logger.info(f"  {col}: {df[col].sum()} cas")

        # Type d'anomalie dominant (pour le RF multiclasse ci-dessous)
        priorite_type = ['anomalie_fraude', 'anomalie_encaissement', 'anomalie_panne',
                          'anomalie_compteur_degrade', 'anomalie_incoherence_spatiale',
                          'anomalie_erreur_releve']
        def type_dominant(row):
            for t in priorite_type:
                if row[t] == 1:
                    return t.replace('anomalie_', '')
            return 'normal'
        df['type_anomalie'] = df[type_cols].apply(
            lambda r: type_dominant(r), axis=1
        )

        return df

    # ------------------------------------------------------------------
    # Methode supervisee : Random Forest
    # ------------------------------------------------------------------
    def _random_forest_typologie(self, df: pd.DataFrame, X: pd.DataFrame) -> pd.DataFrame:
        """[NEW] Random Forest supervise entraine sur les pseudo-labels
        d'anomalie/typologie produits par les etapes precedentes :
        - un classifieur binaire (anomalie / normal) qui sert de 3e score
          dans l'ensemble et dont les probabilites nourrissent le score
          d'alerte ;
        - un classifieur multiclasse pour la TYPOLOGIE (fraude, panne,
          erreur de releve, compteur degrade, encaissement, incoherence
          spatiale, normal), avec importance des variables pour
          l'explicabilite metier demandee par le theme.
        """
        logger.info("Entrainement Random Forest (detection + typologie)...")

        y_bin = df['anomaly_pred_ensemble']
        # Cas degenere (aucune anomalie / que des anomalies) : on saute le RF
        if y_bin.nunique() < 2:
            logger.warning("Random Forest ignore : une seule classe presente dans les pseudo-labels.")
            df['anomaly_score_rf'] = 0.0
            df['anomaly_pred_rf'] = 0
            return df

        # [FIX] meme precaution que pour le RF de typologie : la classe
        # minoritaire (anomalies) peut être trop rare pour stratifier selon
        # le taux de contamination configure.
        stratify_arg = y_bin if (y_bin.value_counts() >= 2).all() else None
        X_train, X_test, y_train, y_test = train_test_split(
            X, y_bin, test_size=self.config.test_size, random_state=RANDOM_STATE, stratify=stratify_arg
        )

        rf_bin = RandomForestClassifier(
            n_estimators=300, max_depth=12, min_samples_leaf=3,
            class_weight='balanced', random_state=RANDOM_STATE, n_jobs=-1
        )
        rf_bin.fit(X_train, y_train)
        y_pred = rf_bin.predict(X_test)
        y_proba = rf_bin.predict_proba(X_test)[:, 1]

        logger.info(f"  RF binaire - accuracy: {accuracy_score(y_test, y_pred):.3f}, "
                     f"f1: {f1_score(y_test, y_pred, zero_division=0):.3f}, "
                     f"AUC: {roc_auc_score(y_test, y_proba) if y_test.nunique() > 1 else float('nan'):.3f}")

        proba_full = rf_bin.predict_proba(X)[:, 1]
        df['anomaly_score_rf'] = proba_full * 100
        df['anomaly_pred_rf'] = (proba_full >= 0.5).astype(int)
        self.models['random_forest_binaire'] = rf_bin

        # Reintegration du RF dans le score d'ensemble (moyenne des 4
        # methodes desormais) et dans la prediction finale.
        df['anomaly_score_ensemble'] = df[
            ['anomaly_score_if', 'anomaly_score_lof', 'anomaly_score_nn', 'anomaly_score_rf']
        ].mean(axis=1)
        pred_cols = ['anomaly_pred_if', 'anomaly_pred_lof', 'anomaly_pred_nn', 'anomaly_pred_rf']
        seuil_vote = max(1, int(np.ceil(len(pred_cols) / 2)))
        df['anomaly_pred_ensemble'] = (df[pred_cols].sum(axis=1) >= seuil_vote).astype(int)

        # --- Classifieur multiclasse pour la typologie ---
        y_type = df['type_anomalie']
        # [FIX] `train_test_split(..., stratify=y_type)` leve un ValueError
        # des qu'une classe (type d'anomalie) compte moins de 2
        # observations - frequent ici car les typologies rares (ex.
        # "erreur_releve") peuvent n'avoir qu'1 ou 2 cas. On stratifie
        # uniquement si chaque classe est suffisamment representee, sinon
        # on bascule sur un split simple non stratifie.
        classes_ok = (y_type.value_counts() >= 2).all()
        if y_type.nunique() > 1:
            if classes_ok:
                X_train2, X_test2, y_train2, y_test2 = train_test_split(
                    X, y_type, test_size=self.config.test_size,
                    random_state=RANDOM_STATE, stratify=y_type
                )
            else:
                logger.warning("Stratification impossible (classe(s) trop rare(s)) : split non stratifie.")
                X_train2, X_test2, y_train2, y_test2 = train_test_split(
                    X, y_type, test_size=self.config.test_size, random_state=RANDOM_STATE
                )
            rf_type = RandomForestClassifier(
                n_estimators=300, max_depth=12, min_samples_leaf=3,
                class_weight='balanced', random_state=RANDOM_STATE, n_jobs=-1
            )
            rf_type.fit(X_train2, y_train2)
            y_pred2 = rf_type.predict(X_test2)
            logger.info(f"  RF typologie - accuracy: {accuracy_score(y_test2, y_pred2):.3f}")
            logger.info("\n" + classification_report(y_test2, y_pred2, zero_division=0))

            df['type_anomalie_predit'] = rf_type.predict(X)
            self.models['random_forest_typologie'] = rf_type

            # Importance des variables (explicabilite - theme : "se basant
            # sur des caracteristiques statistiques et comportementales")
            importance = pd.DataFrame({
                'feature': X.columns,
                'importance': rf_type.feature_importances_
            }).sort_values('importance', ascending=False)

            fig, ax = plt.subplots(figsize=(10, 8))
            importance.head(15).plot(x='feature', y='importance', kind='barh', ax=ax, color='darkorange')
            ax.set_title("Importance des Variables - Typologie des Anomalies (Random Forest)")
            plt.tight_layout()
            plt.savefig(os.path.join(self.config.output_path, 'importance_features_anomalies_rf.png'), dpi=150)
            plt.close()
            importance.to_csv(os.path.join(self.config.output_path, 'importance_features_anomalies_rf.csv'),
                               index=False)
        else:
            df['type_anomalie_predit'] = df['type_anomalie']

        return df

    # ------------------------------------------------------------------
    # Systeme d'alerte (theme point 1)
    # ------------------------------------------------------------------
    def _score_alerte(self, df: pd.DataFrame) -> pd.DataFrame:
        """[NEW] Combine le score d'anomalie (comportement statistique) et
        le score de risque metier (financier + criticite) en un score
        d'alerte unique [0-100], utilise pour prioriser les investigations
        terrain plutot que de traiter toutes les anomalies simultanement."""

        score_risque = df.get('score_risque_global', pd.Series(0, index=df.index))
        df['score_alerte'] = (
            df['anomaly_score_ensemble'] * 0.6 +
            score_risque * 0.4
        ).clip(0, 100)

        df['niveau_alerte'] = pd.cut(
            df['score_alerte'],
            bins=[0, 40, 65, 85, 100],
            labels=['Info', 'Surveillance', 'Prioritaire', 'Urgent'],
            include_lowest=True
        )

        # Priorite d'investigation (1 = a traiter en premier)
        df['priorite_investigation'] = df['score_alerte'].rank(ascending=False, method='first').astype(int)

        n_urgent = (df['niveau_alerte'] == 'Urgent').sum()
        n_prio = (df['niveau_alerte'] == 'Prioritaire').sum()
        logger.info(f"Systeme d'alerte : {n_urgent} cas 'Urgent', {n_prio} cas 'Prioritaire' "
                     f"a investiguer en priorite sur {len(df)} clients")

        # Export de la liste priorisee pour les equipes terrain
        cols_export = [c for c in [
            'code_client', 'score_alerte', 'niveau_alerte', 'priorite_investigation',
            'type_anomalie_predit', 'score_risque_global', 'creance_totale',
            'taux_encaissement', 'zone_geo'
        ] if c in df.columns]
        df.sort_values('priorite_investigation')[cols_export].head(200).to_csv(
            os.path.join(self.config.output_path, 'alertes_priorisees_terrain.csv'), index=False
        )

        return df

    # ------------------------------------------------------------------
    # Visualisations spatio-temporelles
    # ------------------------------------------------------------------
    def _visualiser_anomalies(self, df: pd.DataFrame):
        """[NEW] Graphiques de synthese des anomalies : repartition par
        type, score d'alerte vs risque, incoherence spatiale, et carte
        interactive Folium des zones a risque (theme point 1)."""

        anomalies = df[df['anomaly_pred_ensemble'] == 1]

        fig, axes = plt.subplots(2, 2, figsize=(14, 10))

        type_counts = df.loc[df['type_anomalie_predit'] != 'normal', 'type_anomalie_predit'].value_counts()
        if len(type_counts) > 0:
            type_counts.plot(kind='barh', ax=axes[0, 0], color='firebrick')
        axes[0, 0].set_title('Repartition des Anomalies par Type (Random Forest)')

        axes[0, 1].scatter(df['score_risque_global'], df['anomaly_score_ensemble'],
                            c=df['score_alerte'], cmap='RdYlGn_r', alpha=0.5)
        axes[0, 1].set_xlabel('Score de Risque Metier')
        axes[0, 1].set_ylabel("Score d'Anomalie (ensemble)")
        axes[0, 1].set_title("Score d'Alerte : Risque vs Anomalie")

        zscore_disponible = (
            'ecart_zone_zscore' in df.columns
            and df['ecart_zone_zscore'].std() > 0  # exclut le cas degenere (colonne source non fournie -> tout a 0.0)
        )

        if zscore_disponible:
            df['ecart_zone_zscore'].clip(-5, 5).hist(bins=50, ax=axes[1, 0], color='teal', edgecolor='black')
            axes[1, 0].axvline(2.5, color='red', linestyle='--')
            axes[1, 0].axvline(-2.5, color='red', linestyle='--')
            axes[1, 0].set_title('Incoherence Spatiale (z-score vs zone geographique)')
        else:
            axes[1, 0].axis('off')
            axes[1, 0].text(0.5, 0.5, "Donnee non fournie par SEAL\n(ecart_consommation indisponible)",
                            ha='center', va='center', fontsize=11, color='gray')

        # Serie temporelle synthetique : volume d'alertes par niveau
        if 'niveau_alerte' in df.columns:
            df['niveau_alerte'].value_counts().reindex(
                ['Info', 'Surveillance', 'Prioritaire', 'Urgent']
            ).plot(kind='bar', ax=axes[1, 1], color=['#2ecc71', '#f1c40f', '#e67e22', '#e74c3c'])
            axes[1, 1].set_title("Volume d'Alertes par Niveau de Priorite")

        plt.tight_layout()
        plt.savefig(os.path.join(self.config.output_path, 'anomalies_synthese.png'), dpi=150)
        plt.close()

        if FOLIUM_AVAILABLE and {'x_client', 'y_client'}.issubset(df.columns) and len(anomalies) > 0:
            map_center = [df['x_client'].mean(), df['y_client'].mean()]
            m = folium.Map(location=map_center, zoom_start=12, tiles='CartoDB dark_matter')
            niveau_colors = {'Info': 'green', 'Surveillance': 'yellow', 'Prioritaire': 'orange', 'Urgent': 'red'}
            sample_anom = anomalies.sample(min(2000, len(anomalies)), random_state=RANDOM_STATE)
            for _, row in sample_anom.iterrows():
                color = niveau_colors.get(str(row.get('niveau_alerte')), 'grey')
                CircleMarker(
                    location=[row['x_client'], row['y_client']],
                    radius=4, color=color, fill=True, fill_opacity=0.7,
                    popup=f"Client: {row['code_client']}<br>"
                          f"Type: {row.get('type_anomalie_predit')}<br>"
                          f"Score alerte: {row.get('score_alerte'):.1f}"
                ).add_to(m)
            HeatMap(sample_anom[['x_client', 'y_client']].values, radius=18, blur=12).add_to(m)
            m.save(os.path.join(self.config.output_path, 'carte_anomalies_alertes.html'))
            logger.info("Carte interactive des anomalies/alertes generee")

        logger.info("Visualisations des anomalies generees")

In [20]:
anomaly_detector = AnomalyDetector(CONFIG)
df_anomalies = anomaly_detector.detect(df_features)

2026-09-24 10:05:15,244 [INFO] ============================================================
2026-09-24 10:05:15,244 [INFO] DETECTION D'ANOMALIES
2026-09-24 10:05:15,249 [INFO] ============================================================
2026-09-24 10:05:17,691 [INFO] Isolation Forest : 4210 anomalies
2026-09-24 10:05:21,998 [INFO] LOF : 4210 anomalies
2026-09-24 10:05:22,007 [INFO] Entrainement de l'autoencodeur (Keras/TensorFlow)...
2026-09-24 10:09:36,006 [INFO] Reseau de neurones (autoencodeur) : 4210 anomalies (seuil erreur reconstruction = 0.2240)
2026-09-24 10:09:36,039 [INFO] Ensemble (IF + LOF + NN, vote >= 2/3) : 1726 anomalies detectees
2026-09-24 10:09:36,054 [INFO]   anomalie_panne: 45 cas
2026-09-24 10:09:36,054 [INFO]   anomalie_fraude: 0 cas
2026-09-24 10:09:36,054 [INFO]   anomalie_erreur_releve: 0 cas
2026-09-24 10:09:36,054 [INFO]   anomalie_compteur_degrade: 0 cas
2026-09-24 10:09:36,065 [INFO]   anomalie_encaissement: 919 cas
2026-09-24 10:09:36,065 [INFO]   anomali

## **PARTIE 6 : PRÉDICTION DU TAUX D'ACCESSIBILITÉ (planification tournées)**

Module conserve du pipeline d'origine (regression cross-sectionnelle,
utile a l'optimisation des tournees de la PARTIE 8). Le theme de stage
demande une **prediction de series temporelles** distincte, dediee aux
encaissements/CA/volumes : voir PARTIE 6bis ci-dessous, qui est le
livrable correspondant au point 2 du theme.

In [21]:
# ==============================================================================
# PARTIE 6 : PREDICTION DU TAUX D'ACCESSIBILITE
# ==============================================================================

class AccessibilitePredictor:
    """Predicteur cross-sectionnel du taux d'accessibilite (planification)."""

    def __init__(self, config: PipelineConfig):
        self.config = config
        self.models = {}
        self.metrics = {}

    def prepare_data(self, df: pd.DataFrame) -> Tuple[pd.DataFrame, pd.Series]:
        feature_cols = [
            'diametre_compteur', 'age_compteur', 'indice_attractivite',
            'ecart_consommation', 'densite_compteurs_km', 'indice_criticite',
            'fiabilite_agent', 'taux_acces_zone', 'variabilite_temps',
            'indice_proximite', 'distance_depot_km', 'voisins_500m',
            'zone_geo', 'risque_panne',
            'type_usage'
        ]
        feature_cols = [c for c in feature_cols if c in df.columns]

        X = df[feature_cols].copy()
        y = df['taux_accessibilite'].copy()

        for col in ['type_usage', 'type_facturation', 'zone_geo']:
            if col in X.columns:
                le = LabelEncoder()
                X[col] = le.fit_transform(X[col].astype(str))

        # [FIX] Les NaN residuels dans les features numeriques faisaient
        # planter certains modeles (RandomForest tolere les NaN dans les
        # versions recentes de sklearn mais pas KNN/XGBoost/LightGBM) ->
        # imputation systematique par mediane avant entrainement.
        X = X.fillna(X.median(numeric_only=True))

        return X, y

    def train_models(self, df: pd.DataFrame) -> Dict:
        logger.info("=" * 60)
        logger.info("PREDICTION DU TAUX D'ACCESSIBILITE")
        logger.info("=" * 60)

        X, y = self.prepare_data(df)
        X_train, X_test, y_train, y_test = train_test_split(
            X, y, test_size=self.config.test_size, random_state=RANDOM_STATE
        )

        results = {}

        logger.info("Entrainement Random Forest...")
        rf = RandomForestRegressor(
            n_estimators=200, max_depth=10, min_samples_leaf=5,
            random_state=RANDOM_STATE, n_jobs=-1
        )
        rf.fit(X_train, y_train)
        y_pred_rf = rf.predict(X_test)
        results['Random Forest'] = {
            'model': rf,
            'rmse': np.sqrt(mean_squared_error(y_test, y_pred_rf)),
            'mae': mean_absolute_error(y_test, y_pred_rf),
            'r2': r2_score(y_test, y_pred_rf)
        }

        if XGB_AVAILABLE:
            logger.info("Entrainement XGBoost...")
            xgb = XGBRegressor(n_estimators=200, max_depth=6, learning_rate=0.1,
                                random_state=RANDOM_STATE, n_jobs=-1)
            xgb.fit(X_train, y_train)
            y_pred_xgb = xgb.predict(X_test)
            results['XGBoost'] = {
                'model': xgb,
                'rmse': np.sqrt(mean_squared_error(y_test, y_pred_xgb)),
                'mae': mean_absolute_error(y_test, y_pred_xgb),
                'r2': r2_score(y_test, y_pred_xgb)
            }

        if LGB_AVAILABLE:
            logger.info("Entrainement LightGBM...")
            lgb = LGBMRegressor(n_estimators=200, max_depth=6, learning_rate=0.1,
                                 random_state=RANDOM_STATE, n_jobs=-1)
            lgb.fit(X_train, y_train)
            y_pred_lgb = lgb.predict(X_test)
            results['LightGBM'] = {
                'model': lgb,
                'rmse': np.sqrt(mean_squared_error(y_test, y_pred_lgb)),
                'mae': mean_absolute_error(y_test, y_pred_lgb),
                'r2': r2_score(y_test, y_pred_lgb)
            }

        logger.info("Entrainement KNN...")
        knn = KNeighborsRegressor(n_neighbors=15, weights='distance', n_jobs=-1)
        knn.fit(X_train, y_train)
        y_pred_knn = knn.predict(X_test)
        results['KNN'] = {
            'model': knn,
            'rmse': np.sqrt(mean_squared_error(y_test, y_pred_knn)),
            'mae': mean_absolute_error(y_test, y_pred_knn),
            'r2': r2_score(y_test, y_pred_knn)
        }

        best_model = min(results, key=lambda x: results[x]['rmse'])
        self.models = {k: v['model'] for k, v in results.items()}

        logger.info("Resultats :")
        for name, metrics in results.items():
            logger.info(f"  {name}: RMSE={metrics['rmse']:.4f}, MAE={metrics['mae']:.4f}, R2={metrics['r2']:.4f}")
        logger.info(f"Meilleur modele : {best_model}")

        if hasattr(results[best_model]['model'], 'feature_importances_'):
            importance = pd.DataFrame({
                'feature': X.columns,
                'importance': results[best_model]['model'].feature_importances_
            }).sort_values('importance', ascending=False)

            fig, ax = plt.subplots(figsize=(10, 8))
            importance.head(15).plot(x='feature', y='importance', kind='barh', ax=ax, color='steelblue')
            ax.set_title(f'Importance des Features - {best_model}')
            plt.tight_layout()
            plt.savefig(os.path.join(self.config.output_path, 'importance_features_accessibilite.png'), dpi=150)
            plt.close()

        self.metrics = results

        for name, m in results.items():
            print(f"{name}: RMSE={m['rmse']:.4f}, MAE={m['mae']:.4f}, R2={m['r2']:.4f}")
        return results

    def predict(self, df: pd.DataFrame, model_name: str = None) -> pd.DataFrame:
        if not self.models:
            raise ValueError("Modeles non entraines. Utiliser train_models d'abord.")

        X, _ = self.prepare_data(df)
        if model_name is None:
            # [FIX] `list(self.models.keys())[0]` selectionnait un modele
            # arbitraire (ordre d'insertion), potentiellement pas le
            # meilleur. On selectionne desormais explicitement le modele au
            # RMSE le plus faible si les metriques sont disponibles.
            if self.metrics:
                model_name = min(self.metrics, key=lambda x: self.metrics[x]['rmse'])
            else:
                model_name = list(self.models.keys())[0]

        df = df.copy()
        df['taux_accessibilite_pred'] = self.models[model_name].predict(X)
        df['ecart_accessibilite'] = df['taux_accessibilite'] - df['taux_accessibilite_pred']
        df['sous_estime'] = (df['ecart_accessibilite'] < -0.1).astype(int)

        return df

In [22]:
# Taux d'accessibilite = releves reelles / total de cycles attendus sur
# l'intervalle entre les deux dernieres releves.
is_reel = df_anomalies['type_facturation'].astype(str).eq('REEL')

intervalle_jours = (df_anomalies['DATLCTDRN'] - df_anomalies['DATLCTPRC']).dt.days
nb_cycles_attendus = (intervalle_jours / df_anomalies['cycle_jours']).clip(lower=1)

df_anomalies['taux_accessibilite'] = np.where(is_reel, 1 / nb_cycles_attendus, 0)

# Repli pour les clients sans intervalle exploitable
df_anomalies['taux_accessibilite'] = pd.Series(df_anomalies['taux_accessibilite'], index=df_anomalies.index).where(
    intervalle_jours.notna(),
    df_anomalies.groupby('code_tournee')['taux_accessibilite'].transform('median')
)
df_anomalies['taux_accessibilite'] = df_anomalies['taux_accessibilite'].fillna(df_anomalies['taux_accessibilite'].mean()).clip(0, 1)

In [23]:
df_anomalies['taux_accessibilite'].value_counts()

taux_accessibilite
1.000000    67463
0.000000     4897
0.989011     4019
0.978261     3586
0.967742     1762
0.947368     1043
0.849057      549
0.957447      424
0.937500      178
0.909091       36
0.720000       22
0.708661       16
0.731707       15
0.769231       15
0.857143       15
0.865385       14
0.681818       11
0.750000       11
0.703125       10
0.796460       10
0.762712       10
0.725806        8
0.737705        8
0.927835        7
0.918367        7
0.756303        6
0.803571        5
0.775862        4
0.810811        3
0.891089        2
0.882353        2
0.782609        2
0.818182        2
0.612245        2
0.526316        2
0.486486        1
0.424528        1
0.743802        1
0.900000        1
0.714286        1
0.697674        1
0.825688        1
0.620690        1
0.461538        1
0.361446        1
0.841121        1
0.364372        1
0.608108        1
0.407240        1
0.500000        1
0.873786        1
0.555556        1
0.676692        1
0.481283        1
0.333333 

In [24]:
# # Create 'taux_accessibilite' feature
# df_anomalies['taux_accessibilite'] = 0  # default value

# is_reel = df_anomalies['type_facturation'].eq('REEL')

# accessibility_rate = (
#     is_reel.groupby(df_anomalies['id_agent']).mean()
# )

# df_anomalies['taux_accessibilite'] = df_anomalies['id_agent'].map(accessibility_rate)

In [25]:
df_anomalies['taux_accessibilite'].unique()

array([0.        , 1.        , 0.9375    , 0.96774194, 0.55555556,
       0.94736842, 0.98901099, 0.73170732, 0.70866142, 0.67669173,
       0.77586207, 0.76923077, 0.97826087, 0.72580645, 0.95744681,
       0.76271186, 0.73770492, 0.72      , 0.7826087 , 0.91836735,
       0.64748201, 0.6122449 , 0.4368932 , 0.79646018, 0.80357143,
       0.8490566 , 0.90909091, 0.81818182, 0.85714286, 0.75      ,
       0.81081081, 0.33333333, 0.48128342, 0.68181818, 0.89108911,
       0.703125  , 0.86538462, 0.52631579, 0.36437247, 0.87378641,
       0.5       , 0.88235294, 0.92783505, 0.75630252, 0.40723982,
       0.60810811, 0.82568807, 0.8411215 , 0.4245283 , 0.48648649,
       0.74380165, 0.71428571, 0.69767442, 0.9       , 0.62068966,
       0.46153846, 0.36144578, 0.57324841])

In [26]:
accessibilite_predictor = AccessibilitePredictor(CONFIG)
resultats_accessibilite = accessibilite_predictor.train_models(df_anomalies)
df_accessibilite = accessibilite_predictor.predict(df_anomalies)

2026-09-24 10:09:50,107 [INFO] ============================================================
2026-09-24 10:09:50,109 [INFO] PREDICTION DU TAUX D'ACCESSIBILITE
2026-09-24 10:09:50,111 [INFO] ============================================================
2026-09-24 10:09:50,190 [INFO] Entrainement Random Forest...
2026-09-24 10:09:52,271 [INFO] Entrainement XGBoost...
2026-09-24 10:09:52,637 [INFO] Entrainement LightGBM...
[LightGBM] [Info] Auto-choosing col-wise multi-threading, the overhead of testing was 0.001923 seconds.
You can set `force_col_wise=true` to remove the overhead.
[LightGBM] [Info] Total Bins 572
[LightGBM] [Info] Number of data points in the train set: 67351, number of used features: 7
[LightGBM] [Info] Start training from score 0.937575
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [W

In [27]:
df_accessibilite

,code_client,id_agent,adresse_pntcpg,x_client_original,y_client_original,x_depot,y_depot,code_tournee,cycle_facturation,id_controle,type_usage,type_facturation,diametre_compteur,DATLCTDRN,DATLCTPRC,jour RLV,age_compteur,code_observation,volume_facture,date_plan_action,action_Coupures,action_Envoi E-mail,action_Juridique,action_Mise en Demeure,action_Relance Avocat,...,anomaly_pred_if,anomaly_score_lof,anomaly_pred_lof,anomaly_score_nn,anomaly_pred_nn,anomaly_score_ensemble,anomaly_pred_ensemble,ecart_zone_zscore,anomalie_panne,anomalie_fraude,anomalie_erreur_releve,anomalie_compteur_degrade,anomalie_encaissement,anomalie_incoherence_spatiale,type_anomalie,anomaly_score_rf,anomaly_pred_rf,type_anomalie_predit,score_alerte,niveau_alerte,priorite_investigation,taux_accessibilite,taux_accessibilite_pred,ecart_accessibilite,sous_estime
0,1B19610,SRM576492,SIDI ABDEL,36.675328,2.876816,36.690928,2.854877,1180108,M01,21188826,C4,ESTIMER,15.0,2025-08-27,2025-07-27,31.0,5.0,Introuvable,36,2025-10-08,0,0,0,0,0,...,1,1.809488e-08,0,0.456878,0,30.275965,1,0.0,0,0,0,0,0,0,normal,58.703545,1,encaissement,30.463904,Info,3059,0.000000,0.173636,-0.173636,1
2,1D18084,SRM576492,Haouche KA,36.688150,2.934983,36.690928,2.854877,1180109,M01,21360422,C4,REEL,40.0,2025-09-23,2025-08-27,27.0,5.0,releve,0,2025-11-17,0,0,0,0,0,...,0,4.161815e-09,0,2.774694,1,14.407322,0,0.0,0,0,0,0,0,0,normal,15.325386,0,normal,18.635231,Info,12346,1.000000,0.966336,0.033664,0
5,1D45528,SRM576492,Haouche MA,36.680988,2.911692,36.690928,2.854877,1180109,M01,21418568,C4,REEL,40.0,2025-09-23,2025-08-27,27.0,2.0,releve,260,2025-12-11,0,0,0,0,0,...,1,5.074086e-09,0,1.637511,1,33.817853,1,0.0,0,0,0,0,0,0,normal,81.110277,1,normal,30.290712,Info,3143,1.000000,0.980644,0.019356,0
8,1345669,SRM576492,Rue Des TR,36.682949,2.875660,36.690928,2.854877,1180108,M01,2322302,C3,REEL,15.0,2025-08-27,2025-07-26,32.0,12.0,releve,81,2025-12-22,0,0,0,0,0,...,1,8.178414e-09,0,0.025409,0,17.261462,0,0.0,0,0,0,0,0,0,normal,20.189241,0,normal,20.358507,Info,10255,0.937500,0.860702,0.076798,0
10,1346615,SRM576492,Lotissemen,36.675137,2.890060,36.690928,2.854877,1180108,M01,2323244,C3,REEL,15.0,2025-08-27,2025-07-26,32.0,8.0,releve,0,NaT,0,0,0,0,0,...,0,1.048495e-08,0,0.011766,0,1.808859,0,0.0,0,0,0,0,0,0,normal,0.521902,0,normal,11.085335,Info,66083,0.937500,0.871878,0.065622,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
250026,1E30839,SRM590192,Cité 2058,36.681038,2.885527,36.690928,2.854877,1180262,T01,21501833,C3,REEL,15.0,2026-03-29,2026-03-17,12.0,0.0,releve,0,NaT,0,0,0,0,0,...,0,5.458242e-07,1,0.036639,0,9.666974,0,0.0,0,0,0,0,0,0,normal,7.275280,0,normal,15.800185,Info,18776,1.000000,0.990735,0.009265,0
250027,1E30844,SRM590192,Cité 2058,36.682436,2.883390,36.690928,2.854877,1180263,T01,21501834,C3,REEL,15.0,2026-03-29,2026-03-17,12.0,0.0,releve,0,NaT,0,0,0,0,0,...,0,2.843772e-07,1,0.039983,0,9.299201,0,0.0,0,0,0,0,0,0,normal,5.942234,0,normal,15.579521,Info,19345,1.000000,0.885462,0.114538,0
250028,1E30223,SRM590192,Cité 2110,36.666569,2.902802,36.690928,2.854877,1180221,T01,21501836,C3,REEL,15.0,2026-03-29,2026-03-17,12.0,0.0,releve,0,NaT,0,0,0,0,0,...,0,3.636310e-07,1,0.060718,0,9.414474,0,0.0,0,0,0,0,0,0,normal,5.895726,0,normal,15.648685,Info,19157,1.000000,0.932974,0.067026,0
250156,1346976,SRM590192,Douar BEKH,36.670600,2.891861,36.690928,2.854877,1180018,T01,2323605,C1,REEL,15.0,2026-03-09,2025-12-01,98.0,8.0,releve,26,2025-12-11,0,0,0,0,0,...,0,1.747862e-08,0,0.124674,0,10.718251,0,0.0,0,0,0,0,0,0,normal,4.174212,0,normal,16.942618,Info,16721,0.918367,0.647469,0.270898,0


## **PARTIE 6bis : PRÉVISION DES SÉRIES TEMPORELLES — ENCAISSEMENTS, CA, VOLUMES**

[NEW] Module ajoute pour repondre explicitement au **point 2 du theme de
stage** ("Prevision Automatisee des Tendances Clients") :
- Analyse des series temporelles des encaissements / CA / volumes
  factures pour en degager les tendances.
- Selection et implementation de modeles de prediction pertinents :
  **statistique** (lissage exponentiel Holt-Winters / repli moyenne
  mobile ponderee), **Machine Learning** (Random Forest / XGBoost sur
  variables retardees) et **reseau de neurones** (LSTM si TensorFlow est
  disponible, sinon MLP sur fenetre glissante).
- Pipeline automatise d'entrainement / mise a jour des modeles.
- Tableau de bord de prevision avec analyse des erreurs (incertitude).

Le module d'origine (PARTIE 6) ne modelisait qu'une regression
cross-sectionnelle du taux d'accessibilite : aucune dimension temporelle
n'etait exploitee, alors que la table 5 (encaissements) contient des
dates de transaction. `DataLoader` a ete complete (PARTIE 2) pour
conserver cette granularite transactionnelle (`encaissements_bruts`)
specifiquement pour ce module.

In [28]:
# ==============================================================================
# PARTIE 6bis : PREVISION DES SERIES TEMPORELLES (ENCAISSEMENTS / CA / VOLUMES)
# ==============================================================================

class SeriesTemporellesPredictor:
    """Prevision des tendances d'encaissement / CA / volumes par approche
    statistique, Machine Learning et reseau de neurones, avec pipeline
    d'entrainement automatise et tableau de bord d'evaluation des erreurs."""

    def __init__(self, config: PipelineConfig, freq: str = 'MS', n_lags: int = 6):
        self.config = config
        self.freq = freq            # 'MS' = debut de mois ; 'W' = semaine
        self.n_lags = n_lags
        self.models: Dict[str, Dict] = {}
        self.series: Dict[str, pd.Series] = {}
        self.previsions: Dict[str, pd.DataFrame] = {}

    # ------------------------------------------------------------------
    # Construction des series temporelles
    # ------------------------------------------------------------------
    def build_series(self, encaissements_bruts: Optional[pd.DataFrame],
                      ca_serie: Optional[pd.DataFrame] = None) -> Dict[str, pd.Series]:
        """Agrege les transactions brutes en series temporelles regulieres."""
        series = {}

        if encaissements_bruts is not None and len(encaissements_bruts) > 0:
            s = (encaissements_bruts
                 .set_index('date_encaissement')['montant_encaisse']
                 .resample(self.freq).sum())
            series['encaissements'] = s
            logger.info(f"Serie 'encaissements' construite : {len(s)} points ({self.freq})")

        if ca_serie is not None and 'date_ca' in ca_serie.columns and len(ca_serie) > 0:
            ca_serie = ca_serie.copy()
            ca_serie['date_ca'] = pd.to_datetime(ca_serie['date_ca'], errors='coerce')
            ca_serie = ca_serie.dropna(subset=['date_ca'])
            if len(ca_serie) > 0:
                s_ca = ca_serie.set_index('date_ca')['ca_2025'].resample(self.freq).sum()
                series['ca'] = s_ca
                logger.info(f"Serie 'ca' construite : {len(s_ca)} points ({self.freq})")

        if not series:
            logger.warning("Aucune serie temporelle exploitable (dates de transaction absentes ou "
                            "insuffisantes). La PARTIE 6bis sera ignoree.")

        self.series = series
        return series

    # ------------------------------------------------------------------
    # Feature engineering pour les modeles ML / NN
    # ------------------------------------------------------------------
    def _make_supervised(self, s: pd.Series) -> pd.DataFrame:
        """Transforme une serie en probleme supervise (lags + saisonnalite)."""
        n_lags = min(self.n_lags, max(1, len(s) // 3))
        out = pd.DataFrame({'y': s})
        for lag in range(1, n_lags + 1):
            out[f'lag_{lag}'] = out['y'].shift(lag)
        out['rolling_mean_3'] = out['y'].shift(1).rolling(3, min_periods=1).mean()
        out['mois'] = out.index.month
        out['tendance'] = np.arange(len(out))
        return out.dropna()

    # ------------------------------------------------------------------
    # Modele statistique (lissage exponentiel / moyenne mobile)
    # ------------------------------------------------------------------
    def _fit_statistique(self, s: pd.Series, horizon: int) -> Tuple[np.ndarray, np.ndarray, float]:
        """Retourne (previsions_in_sample_fit, previsions_futures, rmse_in_sample)."""
        if STATSMODELS_AVAILABLE and len(s) >= 8:
            try:
                seasonal_periods = 12 if self.freq == 'MS' and len(s) >= 24 else None
                model = ExponentialSmoothing(
                    s, trend='add',
                    seasonal='add' if seasonal_periods else None,
                    seasonal_periods=seasonal_periods,
                    initialization_method='estimated'
                ).fit()
                fitted = model.fittedvalues
                forecast = model.forecast(horizon)
                rmse = np.sqrt(mean_squared_error(s.iloc[1:], fitted.iloc[1:])) if len(s) > 1 else np.nan
                return fitted.values, forecast.values, rmse
            except Exception as e:
                logger.warning(f"Holt-Winters a echoue ({e}), repli sur moyenne mobile ponderee.")

        # [Repli] Moyenne mobile ponderee (poids lineaires croissants) :
        # methode statistique simple, toujours applicable, meme sur de
        # tres courtes series.
        window = min(3, len(s))
        weights = np.arange(1, window + 1)
        fitted = s.rolling(window).apply(lambda x: np.average(x, weights=weights), raw=True)
        derniere_moy = np.average(s.iloc[-window:], weights=weights)
        forecast = np.repeat(derniere_moy, horizon)
        valid = fitted.dropna()
        rmse = np.sqrt(mean_squared_error(s.loc[valid.index], valid)) if len(valid) > 0 else np.nan
        return fitted.values, forecast, rmse

    # ------------------------------------------------------------------
    # Modele Machine Learning (Random Forest / XGBoost sur variables retardees)
    # ------------------------------------------------------------------
    def _fit_ml(self, s: pd.Series, horizon: int) -> Tuple[object, np.ndarray, np.ndarray, float]:
        sup = self._make_supervised(s)
        if len(sup) < 5:
            logger.warning("Serie trop courte pour un modele ML (< 5 observations utilisables).")
            return None, np.array([]), np.repeat(s.iloc[-1], horizon), np.nan

        X, y = sup.drop(columns='y'), sup['y']
        n_test = max(1, int(len(X) * self.config.test_size))
        X_train, X_test = X.iloc[:-n_test], X.iloc[-n_test:]
        y_train, y_test = y.iloc[:-n_test], y.iloc[-n_test:]

        if XGB_AVAILABLE and len(X_train) >= 10:
            model = XGBRegressor(n_estimators=200, max_depth=4, learning_rate=0.08,
                                  random_state=RANDOM_STATE, n_jobs=-1)
        else:
            model = RandomForestRegressor(n_estimators=300, max_depth=6, min_samples_leaf=2,
                                           random_state=RANDOM_STATE, n_jobs=-1)
        model.fit(X_train, y_train)

        y_pred_test = model.predict(X_test) if len(X_test) > 0 else np.array([])
        rmse = np.sqrt(mean_squared_error(y_test, y_pred_test)) if len(y_test) > 0 else np.nan

        # Prevision multi-pas iterative (feedback des predictions comme lags)
        forecast = self._forecast_iteratif(model, s, horizon)
        return model, y_pred_test, forecast, rmse

    def _forecast_iteratif(self, model, s: pd.Series, horizon: int) -> np.ndarray:
        """Prevision recursive multi-pas : chaque prediction devient le
        dernier lag de l'etape suivante."""
        historique = list(s.values)
        n_lags = min(self.n_lags, max(1, len(s) // 3))
        previsions = []
        for h in range(horizon):
            derniers = historique[-n_lags:]
            row = {f'lag_{i+1}': derniers[-(i + 1)] for i in range(n_lags)}
            row['rolling_mean_3'] = np.mean(historique[-3:])
            date_future = s.index[-1] + (h + 1) * (s.index[1] - s.index[0]) if len(s.index) > 1 else s.index[-1]
            row['mois'] = date_future.month if hasattr(date_future, 'month') else 0
            row['tendance'] = len(historique)
            X_row = pd.DataFrame([row])[[c for c in row.keys()]]
            # Reordonne les colonnes comme a l'entrainement si possible
            pred = model.predict(X_row)[0]
            previsions.append(pred)
            historique.append(pred)
        return np.array(previsions)

    # ------------------------------------------------------------------
    # Reseau de neurones (LSTM si TensorFlow, sinon MLP sur fenetre glissante)
    # ------------------------------------------------------------------
    def _fit_nn(self, s: pd.Series, horizon: int) -> Tuple[object, np.ndarray, np.ndarray, float]:
        n_lags = min(self.n_lags, max(1, len(s) // 3))
        valeurs = s.values.astype(float)
        if len(valeurs) < n_lags + 3:
            logger.warning("Serie trop courte pour un reseau de neurones.")
            return None, np.array([]), np.repeat(s.iloc[-1], horizon), np.nan

        scaler = MinMaxScaler()
        valeurs_scaled = scaler.fit_transform(valeurs.reshape(-1, 1)).flatten()

        X, y = [], []
        for i in range(n_lags, len(valeurs_scaled)):
            X.append(valeurs_scaled[i - n_lags:i])
            y.append(valeurs_scaled[i])
        X, y = np.array(X), np.array(y)

        n_test = max(1, int(len(X) * self.config.test_size))
        X_train, X_test = X[:-n_test], X[-n_test:]
        y_train, y_test = y[:-n_test], y[-n_test:]

        if TF_AVAILABLE and len(X_train) >= 8:
            X_train_lstm = X_train.reshape((X_train.shape[0], X_train.shape[1], 1))
            X_test_lstm = X_test.reshape((X_test.shape[0], X_test.shape[1], 1))

            model = Sequential([
                Input(shape=(n_lags, 1)),
                LSTM(16, activation='tanh', return_sequences=False),
                Dropout(0.1),
                Dense(8, activation='relu'),
                Dense(1)
            ])
            model.compile(optimizer=Adam(learning_rate=1e-3), loss='mse')
            early_stop = EarlyStopping(monitor='loss', patience=10, restore_best_weights=True)
            model.fit(X_train_lstm, y_train, epochs=150, batch_size=8, verbose=0,
                      callbacks=[early_stop])

            y_pred_test_scaled = model.predict(X_test_lstm, verbose=0).flatten() if len(X_test) > 0 else np.array([])
            backend = 'lstm'
        else:
            # [FIX] `early_stopping=True` reserve automatiquement une
            # fraction de X_train pour la validation interne ; sur une
            # serie courte (quelques dizaines de points, frequent pour des
            # encaissements mensuels), cette fraction devient trop petite
            # et sklearn leve un ValueError. On ne l'active que si
            # l'echantillon d'entrainement est suffisant.
            early_stop_ok = len(X_train) >= 20
            model = MLPRegressor(hidden_layer_sizes=(16, 8), activation='relu', solver='adam',
                                  max_iter=800, random_state=RANDOM_STATE,
                                  early_stopping=early_stop_ok)
            model.fit(X_train, y_train)
            y_pred_test_scaled = model.predict(X_test) if len(X_test) > 0 else np.array([])
            backend = 'mlp'

        if len(y_pred_test_scaled) > 0:
            y_pred_test = scaler.inverse_transform(y_pred_test_scaled.reshape(-1, 1)).flatten()
            y_test_real = scaler.inverse_transform(y_test.reshape(-1, 1)).flatten()
            rmse = np.sqrt(mean_squared_error(y_test_real, y_pred_test))
        else:
            y_pred_test, rmse = np.array([]), np.nan

        # Prevision multi-pas iterative sur la fenetre normalisee
        fenetre = list(valeurs_scaled[-n_lags:])
        previsions_scaled = []
        for _ in range(horizon):
            x_in = np.array(fenetre[-n_lags:])
            if backend == 'lstm':
                pred = model.predict(x_in.reshape(1, n_lags, 1), verbose=0)[0, 0]
            else:
                pred = model.predict(x_in.reshape(1, -1))[0]
            previsions_scaled.append(pred)
            fenetre.append(pred)
        forecast = scaler.inverse_transform(np.array(previsions_scaled).reshape(-1, 1)).flatten()

        return model, y_pred_test, forecast, rmse

    # ------------------------------------------------------------------
    # Pipeline automatise (entrainement + prevision + sauvegarde modeles)
    # ------------------------------------------------------------------
    def entrainer_et_prevoir(self, nom_serie: str, s: pd.Series, horizon: int = None) -> pd.DataFrame:
        """Entraine les 3 familles de modeles sur une serie donnee, choisit
        la meilleure par RMSE, et produit les previsions futures avec
        bande d'incertitude. Sauvegarde automatiquement les modeles sur
        disque (pipeline "automatise" demande par le theme : peut etre
        rappelee periodiquement avec des donnees mises a jour)."""

        horizon = horizon or self.config.horizon_prevision_mois
        logger.info("=" * 60)
        logger.info(f"PREVISION SERIE TEMPORELLE : {nom_serie} (horizon={horizon})")
        logger.info("=" * 60)

        if len(s) < 4:
            logger.warning(f"Serie '{nom_serie}' trop courte ({len(s)} points) pour une prevision fiable.")
            return pd.DataFrame()

        resultats = {}

        fitted_stat, forecast_stat, rmse_stat = self._fit_statistique(s, horizon)
        resultats['Statistique (Holt-Winters/MM ponderee)'] = {'forecast': forecast_stat, 'rmse': rmse_stat}

        model_ml, pred_test_ml, forecast_ml, rmse_ml = self._fit_ml(s, horizon)
        resultats['Machine Learning (RF/XGBoost)'] = {'forecast': forecast_ml, 'rmse': rmse_ml, 'model': model_ml}

        model_nn, pred_test_nn, forecast_nn, rmse_nn = self._fit_nn(s, horizon)
        resultats['Reseau de Neurones (LSTM/MLP)'] = {'forecast': forecast_nn, 'rmse': rmse_nn, 'model': model_nn}

        for name, r in resultats.items():
            rmse_str = f"{r['rmse']:.2f}" if pd.notna(r['rmse']) else "N/A"
            logger.info(f"  {name} : RMSE (validation) = {rmse_str}")

        # Selection du meilleur modele (RMSE le plus faible parmi ceux valides)
        rmse_valides = {k: v['rmse'] for k, v in resultats.items() if pd.notna(v['rmse'])}
        meilleur = min(rmse_valides, key=rmse_valides.get) if rmse_valides else list(resultats.keys())[0]
        logger.info(f"Meilleur modele retenu pour '{nom_serie}' : {meilleur}")

        # Bande d'incertitude : +/- 1.96 * RMSE du meilleur modele (approx.
        # gaussienne des erreurs de prevision, +/- ~95%)
        rmse_ref = rmse_valides.get(meilleur, s.std())
        forecast_ref = resultats[meilleur]['forecast']

        dates_futures = pd.date_range(
            start=s.index[-1] + (s.index[1] - s.index[0]) if len(s.index) > 1 else s.index[-1],
            periods=horizon, freq=self.freq
        )
        df_prev = pd.DataFrame({
            'date': dates_futures,
            'prevision': forecast_ref,
            'borne_basse_95': forecast_ref - 1.96 * rmse_ref,
            'borne_haute_95': forecast_ref + 1.96 * rmse_ref,
            'modele_retenu': meilleur
        })
        for name, r in resultats.items():
            df_prev[f'prevision_{name.split(" ")[0].lower()}'] = r['forecast']

        # Sauvegarde (pipeline automatise : modeles + previsions persistes)
        self.models[nom_serie] = resultats
        self.previsions[nom_serie] = df_prev
        with open(os.path.join(self.config.models_path, f'previsions_{nom_serie}.pkl'), 'wb') as f:
            pickle.dump({'resultats': {k: v.get('rmse') for k, v in resultats.items()},
                         'previsions': df_prev}, f)
        df_prev.to_csv(os.path.join(self.config.output_path, f'previsions_{nom_serie}.csv'), index=False)

        self._dashboard(nom_serie, s, df_prev, resultats, rmse_valides)

        return df_prev

    # ------------------------------------------------------------------
    # Tableau de bord (prevision + analyse des erreurs / incertitude)
    # ------------------------------------------------------------------
    def _dashboard(self, nom_serie: str, s: pd.Series, df_prev: pd.DataFrame,
                   resultats: Dict, rmse_valides: Dict):
        fig, axes = plt.subplots(2, 1, figsize=(13, 9), gridspec_kw={'height_ratios': [2, 1]})

        # Historique + previsions des 3 modeles + bande d'incertitude
        axes[0].plot(s.index, s.values, label='Historique', color='black', marker='o', markersize=3)
        colors = {'Statistique (Holt-Winters/MM ponderee)': 'tab:blue',
                  'Machine Learning (RF/XGBoost)': 'tab:green',
                  'Reseau de Neurones (LSTM/MLP)': 'tab:red'}
        for name, r in resultats.items():
            if len(r['forecast']) > 0:
                axes[0].plot(df_prev['date'], r['forecast'], linestyle='--', marker='x',
                             label=f"Prevision - {name}", color=colors.get(name))
        axes[0].fill_between(df_prev['date'], df_prev['borne_basse_95'], df_prev['borne_haute_95'],
                              color='grey', alpha=0.25, label='Intervalle de confiance ~95%')
        axes[0].set_title(f"Prevision de la serie '{nom_serie}' — modele retenu : "
                           f"{df_prev['modele_retenu'].iloc[0]}")
        axes[0].legend(fontsize=8, loc='upper left')
        axes[0].set_ylabel(nom_serie)

        # Comparaison des erreurs (RMSE) par famille de modele
        if rmse_valides:
            noms = list(rmse_valides.keys())
            valeurs = [rmse_valides[n] for n in noms]
            axes[1].barh([n.split(' (')[0] for n in noms], valeurs,
                          color=[colors.get(n, 'grey') for n in noms])
            axes[1].set_xlabel('RMSE (validation)')
            axes[1].set_title("Analyse des erreurs par famille de modele (plus bas = meilleur)")

        plt.tight_layout()
        plt.savefig(os.path.join(self.config.output_path, f'dashboard_prevision_{nom_serie}.png'), dpi=150)
        plt.close()
        logger.info(f"Tableau de bord de prevision genere pour '{nom_serie}'")

In [29]:
serie_predictor = SeriesTemporellesPredictor(CONFIG, freq='MS', n_lags=6)
series_disponibles = serie_predictor.build_series(
    encaissements_bruts=loader.encaissements_bruts,
    ca_serie=loader.tables.get('ca_serie')
)

previsions = {}
for nom, s in series_disponibles.items():
    previsions[nom] = serie_predictor.entrainer_et_prevoir(nom, s)

# Pour rejouer le pipeline automatiquement avec des donnees mises a jour :
#   loader2 = DataLoader(CONFIG); df2 = loader2.load_all()
#   nouvelles_series = serie_predictor.build_series(loader2.encaissements_bruts)
#   for nom, s in nouvelles_series.items(): serie_predictor.entrainer_et_prevoir(nom, s)

2026-09-24 10:09:54,255 [INFO] Serie 'encaissements' construite : 12 points (MS)
2026-09-24 10:09:54,255 [INFO] ============================================================
2026-09-24 10:09:54,255 [INFO] PREVISION SERIE TEMPORELLE : encaissements (horizon=3)
2026-09-24 10:09:54,262 [INFO] ============================================================


c:\Users\djamel\anaconda3\Lib\site-packages\statsmodels\tsa\holtwinters\model.py:918: ConvergenceWarning: Optimization failed to converge. Check mle_retvals.
  warnings.warn(


2026-09-24 10:09:54,956 [INFO]   Statistique (Holt-Winters/MM ponderee) : RMSE (validation) = 2647676.70
2026-09-24 10:09:54,961 [INFO]   Machine Learning (RF/XGBoost) : RMSE (validation) = 1015516.16
2026-09-24 10:09:54,961 [INFO]   Reseau de Neurones (LSTM/MLP) : RMSE (validation) = 129037.14
2026-09-24 10:09:54,961 [INFO] Meilleur modele retenu pour 'encaissements' : Reseau de Neurones (LSTM/MLP)
2026-09-24 10:09:55,484 [INFO] Tableau de bord de prevision genere pour 'encaissements'


## **PARTIE 7 : SEGMENTATION CLIENT STRATÉGIQUE & STRATÉGIES DE RECOUVREMENT**

Alignement avec le theme de stage (point 3 - "Analyse de donnees
multidimensionnelles") :
- Identification des variables cles du comportement de paiement (creance,
  taux d'encaissement, delai de paiement) en plus des variables
  operationnelles deja presentes.
- Reduction de dimensionnalite (ACP) pour la visualisation.
- Clustering (K-Means + GMM) pour regrouper les clients par comportement.
- Definition d'une **strategie de recouvrement differenciee par segment**
  et **suivi de son efficacite** (taux d'encaissement moyen observe par
  segment), au lieu de la seule "priorisation de tournee" du pipeline
  d'origine.

In [30]:
# ==============================================================================
# PARTIE 7 : SEGMENTATION CLIENT ET STRATEGIE DE RECOUVREMENT
# ==============================================================================

class CustomerSegmentation:
    """Segmentation multidimensionnelle des clients pour piloter des
    strategies de recouvrement differenciees et en suivre l'efficacite."""

    def __init__(self, config: PipelineConfig):
        self.config = config
        self.models = {}

    def segment(self, df: pd.DataFrame) -> pd.DataFrame:
        """Pipeline de segmentation."""
        logger.info("=" * 60)
        logger.info("SEGMENTATION CLIENT & STRATEGIE DE RECOUVREMENT")
        logger.info("=" * 60)

        # Features de segmentation : dimensions operationnelles (tournee)
        # ET dimensions de comportement de paiement / recouvrement.
        # [FIX/NEW] Le pipeline d'origine utilisait deja 'creance_totale' et
        # 'taux_encaissement', mais aucune variable ne capturait la
        # regularite de paiement dans le temps -> ajout de
        # 'delai_dernier_encaissement', 'poids_creance_ca' et
        # 'retard_relatif_paiement' (crees en PARTIE 3).

        # seg_features = [
        #     'taux_accessibilite', 'indice_attractivite', 'indice_criticite',
        #     'score_risque_global', 'urgence_releve', 'distance_depot_km',
        #     'temps_service', 'productivite_attendue',
        #     'creance_totale', 'taux_encaissement', 'delai_dernier_encaissement',
        #     'poids_creance_ca', 'retard_relatif_paiement',
        #     'age_compteur', 'densite_compteurs_km'
        # ]

        seg_features = [
            'taux_accessibilite', 'indice_attractivite', 'indice_criticite',
            'score_risque_global', 'urgence_releve', 'distance_depot_km',
            'creance_totale', 'taux_encaissement', 'delai_dernier_encaissement',
            'poids_creance_ca', 'retard_relatif_paiement',
            'age_compteur', 'densite_compteurs_km'
            # 'temps_service' retire : constante degeneree (SEAL n'a pas fourni
            # temps_service / temps_trajet, valeur = 15 pour 100% des clients)
        ]

        seg_features = [c for c in seg_features if c in df.columns]

        X = df[seg_features].fillna(df[seg_features].median())
        scaler = StandardScaler()
        X_scaled = scaler.fit_transform(X)

        # ACP pour reduction de dimensionnalite / visualisation
        pca = PCA(n_components=2)
        X_pca = pca.fit_transform(X_scaled)
        logger.info(f"ACP : {pca.explained_variance_ratio_.sum():.1%} de variance expliquee (2 composantes)")

        # Selection du nombre de clusters (methode du coude + silhouette)
        inertias, silhouettes = [], []
        k_range = range(2, 10)
        for k in k_range:
            km = KMeans(n_clusters=k, random_state=RANDOM_STATE, n_init=10)
            labels = km.fit_predict(X_scaled)
            inertias.append(km.inertia_)
            silhouettes.append(silhouette_score(X_scaled, labels))

        fig, axes = plt.subplots(1, 2, figsize=(12, 4))
        axes[0].plot(k_range, inertias, 'bo-')
        axes[0].set_title('Methode du Coude')
        axes[1].plot(k_range, silhouettes, 'ro-')
        axes[1].set_title('Score de Silhouette')
        plt.tight_layout()
        plt.savefig(os.path.join(self.config.output_path, 'segmentation_selection_k.png'), dpi=150)
        plt.close()

        # [FIX] Le nombre de clusters final restait fige a `config.n_clusters`
        # sans tenir compte du resultat de la recherche ci-dessus. On
        # retient desormais le k qui maximise le score de silhouette parmi
        # la plage testee, sauf si l'utilisateur a explicitement fixe
        # `n_clusters_defaut` a une valeur differente de la config par
        # defaut (dans ce cas on respecte son choix).
        # k_optimal = list(k_range)[int(np.argmax(silhouettes))]
        # n_clusters = k_optimal if self.config.n_clusters == 5 else self.config.n_clusters

        n_clusters = 5

        logger.info(f"Nombre de clusters retenu : {n_clusters} "
                     f"(silhouette optimale sur la plage testee : k={n_clusters})")



        kmeans = KMeans(n_clusters=n_clusters, random_state=RANDOM_STATE, n_init=10)
        df = df.copy()
        df['segment'] = kmeans.fit_predict(X_scaled)

        gmm = GaussianMixture(n_components=n_clusters, random_state=RANDOM_STATE, n_init=10)
        gmm.fit(X_scaled)
        df['segment_prob'] = gmm.predict_proba(X_scaled).max(axis=1)

        profile = self._profile_segments(df)

        fig, ax = plt.subplots(figsize=(10, 8))
        scatter = ax.scatter(X_pca[:, 0], X_pca[:, 1], c=df['segment'],
                              cmap='tab10', alpha=0.6, s=50)
        ax.set_xlabel(f'PC1 ({pca.explained_variance_ratio_[0]:.1%})')
        ax.set_ylabel(f'PC2 ({pca.explained_variance_ratio_[1]:.1%})')
        ax.set_title('Segments Clients - Recouvrement (ACP)')
        plt.colorbar(scatter, ax=ax, label='Segment')
        plt.tight_layout()
        plt.savefig(os.path.join(self.config.output_path, 'segmentation_clients.png'), dpi=150)
        plt.close()

        # [NEW] Suivi de l'efficacite des strategies par segment
        self._suivi_efficacite_strategies(df, profile)

        self.models = {'scaler': scaler, 'pca': pca, 'kmeans': kmeans, 'gmm': gmm}
        logger.info("Segmentation et strategie de recouvrement terminees")
        return df

    def _profile_segments(self, df: pd.DataFrame) -> pd.DataFrame:
        """Profilage des segments et attribution d'une strategie de
        recouvrement differenciee (theme : "adapter les strategies... et
        mesurer si ces strategies fonctionnent")."""

        agg_candidats = {
            'taux_accessibilite': 'mean',
            'score_risque_global': 'mean',
            'urgence_releve': 'mean',
            # 'productivite_attendue': 'mean',
            'distance_depot_km': 'mean',
            'creance_totale': 'mean',
            'taux_encaissement': 'mean',
            'delai_dernier_encaissement': 'mean',
            'age_compteur': 'mean',
            # 'temps_service': 'mean',
            'code_client': 'count'
        }
        agg_dict = {k: v for k, v in agg_candidats.items() if k in df.columns}

        profile = df.groupby('segment').agg(agg_dict).reset_index()
        profile = profile.rename(columns={'code_client': 'nb_clients'})
        profile['pct_clients'] = (profile['nb_clients'] / len(df) * 100).round(1)

        # [NEW] Nommage des segments ET strategie de recouvrement associee.
        # La logique d'origine ne nommait que le profil operationnel
        # ("Urgent", "Performant"...) sans jamais relier le segment a une
        # action de recouvrement concrete, alors que c'est l'objet meme du
        # point 3 du theme.
        segment_names, strategies = {}, {}
        for _, row in profile.iterrows():
            seg = int(row['segment'])
            creance = row.get('creance_totale', 0)
            taux_enc = row.get('taux_encaissement', 1)
            risque = row.get('score_risque_global', 0)
            delai = row.get('delai_dernier_encaissement', 0)

            if risque > 60 and taux_enc < 0.5:
                name = "Critique - Fort risque de non-paiement"
                strategie = "Mise en demeure + coupure operationnelle prioritaire"
            elif creance > profile['creance_totale'].quantile(0.75) and taux_enc < 0.7:
                name = "Debiteur chronique - Creance elevee"
                strategie = "Relance avocat / plan d'apurement formel"
            elif delai >= 990:
                # [FIX] delai proche de 999 = valeur de repli (Chapitre 3) indiquant
                # une ABSENCE totale d'encaissement enregistre, non un simple retard
                # progressif. La regle "Retardataire" ci-dessous suppose un
                # comportement de paiement irregulier mais existant ; elle est donc
                # trompeuse pour ce cas et doit etre evaluee AVANT elle.
                name = "Sans historique de paiement - Creance moderee"
                strategie = "Verification du dossier client + premiere relance formelle"
            elif delai > profile['delai_dernier_encaissement'].median() and taux_enc < 0.8:
                name = "Retardataire - Paiement irregulier"
                strategie = "Relance SMS/Appel CATO + suivi rapproche"
            elif taux_enc >= 0.9 and risque < 30:
                name = "Bon payeur - Faible risque"
                strategie = "Suivi standard, pas d'action de recouvrement renforcee"
            elif row.get('taux_accessibilite', 1) < 0.1:
                name = "Inaccessible - Releve non a jour"
                strategie = "Intervention technique / verification terrain avant relance"
            else:
                name = "Standard - Comportement regulier"
                strategie = "Relance preventive legere (GPP Hors APC)"

            segment_names[seg] = name
            strategies[seg] = strategie


        noms_comptes = pd.Series(list(segment_names.values())).value_counts()
        noms_dupliques = noms_comptes[noms_comptes > 1].index.tolist()

        if noms_dupliques:
            compteur_par_nom = {n: 0 for n in noms_dupliques}
            for seg in sorted(segment_names.keys()):
                nom = segment_names[seg]
                if nom in noms_dupliques:
                    compteur_par_nom[nom] += 1
                    segment_names[seg] = f"{nom} (groupe {compteur_par_nom[nom]})"


        df['segment_nom'] = df['segment'].map(segment_names)
        df['strategie_recouvrement'] = df['segment'].map(strategies)
        profile['nom_segment'] = profile['segment'].map(segment_names)
        profile['strategie_recouvrement'] = profile['segment'].map(strategies)

        logger.info(f"Profil des segments :\n{profile.to_string()}")
        profile.to_csv(os.path.join(self.config.output_path, 'profil_segments.csv'), index=False)

        return profile

    def _suivi_efficacite_strategies(self, df: pd.DataFrame, profile: pd.DataFrame):
        """[NEW] Suivi de l'efficacite des strategies de recouvrement par
        segment : le taux d'encaissement moyen sert de proxy de
        performance, mis en regard du niveau de creance et du volume de
        clients concernes (theme : "mesurer si ces strategies
        fonctionnent")."""

        if not {'taux_encaissement', 'strategie_recouvrement'}.issubset(df.columns):
            return

        efficacite = df.groupby(['segment', 'segment_nom', 'strategie_recouvrement']).agg(
            nb_clients=('code_client', 'count'),
            taux_encaissement_moyen=('taux_encaissement', 'mean'),
            creance_totale_segment=('creance_totale', 'sum'),
        ).reset_index().sort_values('taux_encaissement_moyen')

        efficacite.to_csv(os.path.join(self.config.output_path, 'efficacite_strategies_recouvrement.csv'),
                           index=False)

        fig, axes = plt.subplots(1, 2, figsize=(14, 6))
        axes[0].barh(efficacite['segment_nom'], efficacite['taux_encaissement_moyen'], color='seagreen')
        axes[0].set_xlabel("Taux d'encaissement moyen")
        axes[0].set_title("Efficacite du recouvrement par segment")
        axes[0].axvline(self.config.seuil_taux_encaissement_bas, color='red', linestyle='--',
                         label='Seuil bas')
        axes[0].legend()

        axes[1].scatter(efficacite['creance_totale_segment'], efficacite['taux_encaissement_moyen'],
                         s=efficacite['nb_clients'] / max(efficacite['nb_clients'].max(), 1) * 800 + 20,
                         c=range(len(efficacite)), cmap='tab10', alpha=0.7)
        for _, row in efficacite.iterrows():
            axes[1].annotate(row['segment_nom'], (row['creance_totale_segment'], row['taux_encaissement_moyen']),
                              fontsize=7, alpha=0.8)
        axes[1].set_xlabel('Creance totale du segment')
        axes[1].set_ylabel("Taux d'encaissement moyen")
        axes[1].set_title('Priorisation : creance vs efficacite de recouvrement (taille = nb clients)')

        plt.tight_layout()
        plt.savefig(os.path.join(self.config.output_path, 'efficacite_strategies_recouvrement.png'), dpi=150)
        plt.close()
        logger.info("Suivi d'efficacite des strategies de recouvrement genere")

In [31]:
segmentation = CustomerSegmentation(CONFIG)
df_segments = segmentation.segment(df_accessibilite)

2026-09-24 10:09:55,532 [INFO] ============================================================
2026-09-24 10:09:55,538 [INFO] SEGMENTATION CLIENT & STRATEGIE DE RECOUVREMENT
2026-09-24 10:09:55,538 [INFO] ============================================================
2026-09-24 10:09:55,607 [INFO] ACP : 63.6% de variance expliquee (2 composantes)
2026-09-24 10:23:26,513 [INFO] Nombre de clusters retenu : 5 (silhouette optimale sur la plage testee : k=5)
2026-09-24 10:24:05,634 [INFO] Profil des segments :
   segment  taux_accessibilite  score_risque_global  urgence_releve  distance_depot_km  creance_totale  taux_encaissement  delai_dernier_encaissement  age_compteur  nb_clients  pct_clients                                    nom_segment                                       strategie_recouvrement
0        0            0.790588            55.062947        0.550629           4.077774    47260.205314           0.001121                  998.429297      7.274805        5120          6.1         

## **PARTIE 8 : OPTIMISATION DES TOURNÉES (TSP simplifié)**

In [32]:
# ==============================================================================
# PARTIE 8 : OPTIMISATION DES TOURNEES (TSP simplifie)
# ==============================================================================

from scipy.spatial import cKDTree


class TourneeOptimizer:
    """Optimisation des tournees par zone et priorite (glouton plus-proche-voisin)."""

    def __init__(self, config: PipelineConfig):
        self.config = config

    def optimize_zone(self, df: pd.DataFrame, zone_id: int = None) -> pd.DataFrame:
        """Optimise l'ordre de visite pour une zone geographique."""
        if zone_id is not None:
            df_zone = df[df['zone_geo'] == zone_id].copy()
        else:
            df_zone = df.copy()

        # if len(df_zone) < 3:
        #     return df_zone

        if len(df_zone) < 3:
            df_zone = df_zone.sort_values(['urgence_releve', 'score_risque_global'], ascending=[False, False])
            df_zone = df_zone.reset_index(drop=True)
            df_zone['ordre_visite'] = range(1, len(df_zone) + 1)
            distances = [0.0]
            for i in range(1, len(df_zone)):
                d = np.sqrt(
                    (df_zone.iloc[i]['x_client'] - df_zone.iloc[i-1]['x_client'])**2 +
                    (df_zone.iloc[i]['y_client'] - df_zone.iloc[i-1]['y_client'])**2
                ) * 111
                distances.append(d)
            df_zone['distance_etape_km'] = distances
            df_zone['distance_cumulee_km'] = np.cumsum(distances)
            df_zone['temps_cumule_min'] = np.cumsum(df_zone['temps_total_estime'])
            return df_zone

        df_zone = df_zone.sort_values(['urgence_releve', 'score_risque_global'], ascending=[False, False])
        df_zone = df_zone.reset_index(drop=True)

        # coords = df_zone[['x_client', 'y_client']].values
        # visited = [0]
        # remaining = set(range(1, len(coords)))

        # while remaining:
        #     last = visited[-1]
        #     nearest = min(remaining,
        #                   key=lambda i: (coords[i, 0] - coords[last, 0]) ** 2 +
        #                                 (coords[i, 1] - coords[last, 1]) ** 2)
        #     visited.append(nearest)
        #     remaining.remove(nearest)


        coords = df_zone[['x_client', 'y_client']].values
        tree = cKDTree(coords)
        visited = [0]
        remaining_mask = np.ones(len(coords), dtype=bool)
        remaining_mask[0] = False

        for _ in range(len(coords) - 1):
            last = coords[visited[-1]]
            k = 1
            while True:
                dist, idx = tree.query(last, k=k+1)
                idx = np.atleast_1d(idx)
                candidates = idx[remaining_mask[idx]]
                if len(candidates) > 0:
                    nearest = candidates[0]
                    break
                k += 1
            visited.append(nearest)
            remaining_mask[nearest] = False




        df_zone = df_zone.iloc[visited].reset_index(drop=True)
        df_zone['ordre_visite'] = range(1, len(df_zone) + 1)

        distances = [0.0]
        for i in range(1, len(df_zone)):
            d = np.sqrt(
                (df_zone.iloc[i]['x_client'] - df_zone.iloc[i - 1]['x_client']) ** 2 +
                (df_zone.iloc[i]['y_client'] - df_zone.iloc[i - 1]['y_client']) ** 2
            ) * 111
            distances.append(d)

        df_zone['distance_etape_km'] = distances
        df_zone['distance_cumulee_km'] = np.cumsum(distances)
        df_zone['temps_cumule_min'] = np.cumsum(df_zone['temps_total_estime'])

        return df_zone

    def generate_routes(self, df: pd.DataFrame) -> Dict[int, pd.DataFrame]:
        """Genere les routes optimisees par zone."""
        logger.info("=" * 60)
        logger.info("OPTIMISATION DES TOURNEES")
        logger.info("=" * 60)

        routes = {}
        zones = df['zone_geo'].unique()

        for zone in zones:
            # if zone == -1:  # Bruit DBSCAN
            #     continue

            route = self.optimize_zone(df, zone)
            if len(route) == 0:
                continue
            routes[zone] = route

            logger.info(f"Zone {zone}: {len(route)} clients, "
                        f"{route['distance_cumulee_km'].iloc[-1]:.2f} km, "
                        f"{route['temps_cumule_min'].iloc[-1]:.0f} min")

        self._visualize_routes(routes)
        return routes

    def _visualize_routes(self, routes: Dict[int, pd.DataFrame]):
        """Visualise les routes sur carte."""
        if not FOLIUM_AVAILABLE or not routes:
            return

        all_data = pd.concat(routes.values())
        map_center = [all_data['x_client'].mean(), all_data['y_client'].mean()]
        m = folium.Map(location=map_center, zoom_start=12, tiles='CartoDB positron')

        colors = ['red', 'blue', 'green', 'purple', 'orange', 'darkred',
                  'darkblue', 'darkgreen', 'cadetblue', 'darkpurple']

        for i, (zone, route) in enumerate(routes.items()):
            color = colors[i % len(colors)]
            for _, row in route.iterrows():
                folium.CircleMarker(
                    location=[row['x_client'], row['y_client']],
                    radius=4, color=color, fill=True, fill_opacity=0.7,
                    popup=f"Ordre: {int(row['ordre_visite'])}<br>"
                          f"Client: {row['code_client']}<br>"
                          f"Urgence: {row['urgence_releve']:.2f}"
                ).add_to(m)

            points = route[['x_client', 'y_client']].values.tolist()
            AntPath(points, color=color, weight=3, opacity=0.7, dash_array=[10, 20]).add_to(m)

        m.save(os.path.join(self.config.output_path, 'carte_routes_optimisees.html'))
        logger.info("Carte des routes optimisees generee")

In [33]:
print(df_features['zone_geo'].value_counts())

zone_geo
1    32183
0    18188
3    12869
2    10997
4     9952
Name: count, dtype: int64


In [34]:
clients_a_exclure = ['1602267', '1604752', '1613863', '1601770', '1846498', '1971292', '1971282']
df_segments = df_segments[~df_segments['code_client'].isin(clients_a_exclure)]

In [35]:
x_cible = df_segments.loc[df_segments['code_client'] == '1971291', 'x_client'].values[0]
y_cible = df_segments.loc[df_segments['code_client'] == '1971291', 'y_client'].values[0]

df_segments = df_segments[~((df_segments['x_client'] == x_cible) & (df_segments['y_client'] == y_cible))]

In [36]:
tournee_optimizer = TourneeOptimizer(CONFIG)
routes_optimisees = tournee_optimizer.generate_routes(df_segments)

2026-09-24 10:24:09,685 [INFO] ============================================================
2026-09-24 10:24:09,685 [INFO] OPTIMISATION DES TOURNEES
2026-09-24 10:24:09,685 [INFO] ============================================================
2026-09-24 10:24:46,939 [INFO] Zone 4: 9800 clients, 57.61 km, 147000 min
2026-09-24 10:25:16,666 [INFO] Zone 2: 10997 clients, 61.75 km, 164955 min
2026-09-24 10:28:37,349 [INFO] Zone 1: 32181 clients, 61.43 km, 482715 min
2026-09-24 10:29:05,322 [INFO] Zone 3: 12869 clients, 36.21 km, 193035 min
2026-09-24 10:29:58,399 [INFO] Zone 0: 18188 clients, 44.96 km, 272820 min
2026-09-24 10:31:43,263 [INFO] Carte des routes optimisees generee


## **PARTIE 9 : PERFORMANCE DES PLANS D'ACTION (PDA) ET IMPACT SUR LE RECOUVREMENT**

[NEW] Cette partie evalue l'efficacite reelle de chaque levier de
recouvrement (colonnes `action_*` issues de la Table 2 - Plans d'Action)
a deux niveaux :

1. **Comparaison brute** (traite vs non-traite) par levier, avec test
   statistique non parametrique (Mann-Whitney U, coherent avec les tests
   de Kruskal-Wallis deja utilises sur ces donnees) - utile pour un
   premier diagnostic, MAIS **biaisee par la politique de ciblage** :
   les leviers les plus severes (Mise en Demeure, Coupures, Juridique)
   sont assignes aux clients deja les plus a risque, ce qui peut faire
   paraitre ces leviers "moins efficaces" alors qu'ils s'appliquent a une
   population plus difficile au depart. Ce biais est signale explicitement
   plutot que d'etre ignore.
2. **Effet ajuste (T-learner)** : deux modeles Random Forest par levier
   (un entraine uniquement sur les clients traites, un uniquement sur les
   non-traites) permettent d'estimer, pour CHAQUE client, l'effet
   individuel (ITE) du levier "toutes choses egales par ailleurs"
   (a score de risque, creance, CA, autres actions... comparables). C'est
   cet effet ajuste qui sert de base a la PARTIE 10 (optimisation).

In [37]:
# ==============================================================================
# PARTIE 9 : PERFORMANCE DES PDA ET IMPACT SUR LE RECOUVREMENT
# ==============================================================================

class PDAPerformanceAnalyzer:
    """Evalue l'efficacite des leviers de recouvrement (PDA) et estime leur
    effet ajuste sur le taux d'encaissement via une approche T-learner."""

    OUTCOME_COLS = ['taux_encaissement', 'montant_encaisse', 'delai_dernier_encaissement', 'creance_totale']

    def __init__(self, config: PipelineConfig):
        self.config = config
        self.action_cols: List[str] = []
        self.models_uplift: Dict[str, Dict] = {}

    # ------------------------------------------------------------------
    def analyze(self, df: pd.DataFrame) -> Tuple[pd.DataFrame, pd.DataFrame, pd.DataFrame]:
        logger.info("=" * 60)
        logger.info("PERFORMANCE DES PDA ET IMPACT SUR LE RECOUVREMENT")
        logger.info("=" * 60)

        df = df.copy()
        self.action_cols = [c for c in df.columns if c.startswith('action_')]
        if not self.action_cols:
            logger.warning("Aucune colonne 'action_*' trouvee : PARTIE 9 ignoree.")
            return df, pd.DataFrame(), pd.DataFrame()

        logger.info(f"Leviers PDA detectes : {[c.replace('action_', '') for c in self.action_cols]}")

        naive_df = self._comparaison_brute(df)
        df, uplift_df = self._effet_ajuste_t_learner(df)

        self._plot_performance(naive_df, uplift_df)

        return df, naive_df, uplift_df

    # ------------------------------------------------------------------
    def _comparaison_brute(self, df: pd.DataFrame) -> pd.DataFrame:
        """Comparaison traite/non-traite par levier (diagnostic rapide,
        biais de ciblage explicitement signale - voir PARTIE 9 en-tete)."""
        logger.info("Comparaison brute traite vs non-traite (par levier)...")

        lignes = []
        for col in self.action_cols:
            treated = df[col] > 0
            if treated.sum() < 10 or (~treated).sum() < 10:
                logger.info(f"  {col} : effectif insuffisant, ignore (traites={treated.sum()})")
                continue

            row = {'levier': col.replace('action_', ''), 'nb_clients_traites': int(treated.sum())}
            for oc in self.OUTCOME_COLS:
                if oc not in df.columns:
                    continue
                v_t = df.loc[treated, oc].dropna()
                v_c = df.loc[~treated, oc].dropna()
                row[f'{oc}_traite'] = v_t.mean()
                row[f'{oc}_non_traite'] = v_c.mean()
                try:
                    _, pval = stats.mannwhitneyu(v_t, v_c, alternative='two-sided')
                except ValueError:
                    pval = np.nan
                row[f'{oc}_pvalue'] = pval
            lignes.append(row)

        naive_df = pd.DataFrame(lignes)
        if not naive_df.empty and 'montant_encaisse_traite' in naive_df.columns:
            naive_df['ecart_montant_encaisse'] = naive_df['montant_encaisse_traite'] - naive_df['montant_encaisse_non_traite']
            naive_df = naive_df.sort_values('ecart_montant_encaisse', ascending=False)

        naive_df.to_csv(os.path.join(self.config.output_path, 'pda_comparaison_brute.csv'), index=False)
        logger.info("  /!\\ Comparaison brute : biais de ciblage possible (leviers durs assignes "
                     "aux clients deja les plus a risque) - voir effet ajuste ci-dessous pour un "
                     "diagnostic corrige de ce biais.")
        return naive_df

    # ------------------------------------------------------------------
    def _effet_ajuste_t_learner(self, df: pd.DataFrame) -> Tuple[pd.DataFrame, pd.DataFrame]:
        """Estime l'effet individuel (ITE) de chaque levier sur le taux
        d'encaissement via un T-learner (deux Random Forest par levier :
        un sur les clients traites, un sur les non-traites), en controlant
        pour le profil de risque/financier du client - ce qui corrige (au
        moins partiellement) le biais de ciblage de la comparaison brute."""
        logger.info("Estimation de l'effet ajuste par levier (T-learner, Random Forest)...")

        outcome_col = 'taux_encaissement'
        covariates_base = [
            'score_risque_global', 'creance_totale', 'ca_2025', 'delai_dernier_encaissement',
            'age_compteur', 'indice_criticite', 'poids_creance_ca', 'retard_relatif_paiement',
            'risque_non_paiement',
        ]
        covariates_base = [c for c in covariates_base if c in df.columns]

        resultats = []
        for col in self.action_cols:
            treated = (df[col] > 0).astype(int)
            if treated.sum() < 30 or (len(df) - treated.sum()) < 30:
                logger.info(f"  {col} : effectif insuffisant pour le T-learner (traites={treated.sum()}), ignore.")
                continue

            autres_leviers = [c for c in self.action_cols if c != col]
            X_cols = covariates_base + autres_leviers
            X = df[X_cols].fillna(df[X_cols].median(numeric_only=True))
            y = df[outcome_col]

            model_traite = RandomForestRegressor(n_estimators=200, max_depth=8, min_samples_leaf=10,
                                                  random_state=RANDOM_STATE, n_jobs=-1)
            model_non_traite = RandomForestRegressor(n_estimators=200, max_depth=8, min_samples_leaf=10,
                                                      random_state=RANDOM_STATE, n_jobs=-1)
            model_traite.fit(X[treated == 1], y[treated == 1])
            model_non_traite.fit(X[treated == 0], y[treated == 0])

            pred_traite = model_traite.predict(X)
            pred_non_traite = model_non_traite.predict(X)
            ite = pred_traite - pred_non_traite  # effet individuel estime

            ite_col = f'ite_{col}'
            df[ite_col] = ite
            self.models_uplift[col] = {'traite': model_traite, 'non_traite': model_non_traite}

            ate_global = float(np.mean(ite))
            resultats.append({
                'levier': col.replace('action_', ''),
                'ate_global_taux_encaissement': ate_global,
                'gain_potentiel_estime_DA': float(np.sum(ite * df['ca_2025'].clip(lower=0))),
                'nb_clients_traites': int(treated.sum()),
            })
            logger.info(f"  {col} : ATE (ajuste) = {ate_global:+.4f} point de taux d'encaissement")

        uplift_df = pd.DataFrame(resultats).sort_values('ate_global_taux_encaissement', ascending=False)
        uplift_df.to_csv(os.path.join(self.config.output_path, 'pda_effet_ajuste_t_learner.csv'), index=False)

        # Heatmap ATE par segment x levier (si segmentation deja disponible)
        if 'segment_nom' in df.columns and resultats:
            ite_cols = [f"ite_{r['levier'] if r['levier'].startswith('action_') else 'action_' + r['levier']}"
                        for r in resultats]
            ite_cols = [c for c in ite_cols if c in df.columns]
            if ite_cols:
                heat = df.groupby('segment_nom')[ite_cols].mean()
                heat.columns = [c.replace('ite_action_', '') for c in heat.columns]
                heat.to_csv(os.path.join(self.config.output_path, 'pda_ate_par_segment.csv'))

                fig, ax = plt.subplots(figsize=(max(8, len(heat.columns) * 1.3), max(4, len(heat) * 0.6)))
                sns.heatmap(heat, annot=True, fmt='.3f', cmap='RdYlGn', center=0, ax=ax,
                            cbar_kws={'label': "Effet estime sur le taux d'encaissement"})
                ax.set_title("Effet ajuste (ATE) de chaque levier PDA, par segment de recouvrement")
                plt.tight_layout()
                plt.savefig(os.path.join(self.config.output_path, 'pda_heatmap_segment_levier.png'), dpi=150)
                plt.close()

        return df, uplift_df

    # ------------------------------------------------------------------
    def _plot_performance(self, naive_df: pd.DataFrame, uplift_df: pd.DataFrame):
        if naive_df.empty and uplift_df.empty:
            return

        fig, axes = plt.subplots(1, 2, figsize=(15, 6))

        if not naive_df.empty and 'taux_encaissement_traite' in naive_df.columns:
            x = np.arange(len(naive_df))
            width = 0.35
            axes[0].bar(x - width / 2, naive_df['taux_encaissement_traite'], width,
                        label='Clients traites', color='firebrick')
            axes[0].bar(x + width / 2, naive_df['taux_encaissement_non_traite'], width,
                        label='Clients non traites', color='steelblue')
            axes[0].set_xticks(x)
            axes[0].set_xticklabels(naive_df['levier'], rotation=30, ha='right')
            axes[0].set_ylabel("Taux d'encaissement moyen")
            axes[0].set_title("Comparaison BRUTE (biais de ciblage possible)")
            axes[0].legend()

        if not uplift_df.empty:
            colors = ['seagreen' if v > 0 else 'firebrick' for v in uplift_df['ate_global_taux_encaissement']]
            axes[1].barh(uplift_df['levier'], uplift_df['ate_global_taux_encaissement'], color=colors)
            axes[1].axvline(0, color='black', linewidth=0.8)
            axes[1].set_xlabel("Effet ajuste (ATE) sur le taux d'encaissement")
            axes[1].set_title("Effet AJUSTE (T-learner) : impact reel estime par levier")

        plt.tight_layout()
        plt.savefig(os.path.join(self.config.output_path, 'pda_performance_comparaison.png'), dpi=150)
        plt.close()
        logger.info("Graphiques de performance PDA generes (brut vs ajuste)")

In [38]:
pda_analyzer = PDAPerformanceAnalyzer(CONFIG)
df_pda, pda_naif, pda_uplift = pda_analyzer.analyze(df_segments)

if not pda_uplift.empty:
    print("\nClassement des leviers par effet ajuste sur le taux d'encaissement :")
    print(pda_uplift[['levier', 'ate_global_taux_encaissement', 'gain_potentiel_estime_DA',
                       'nb_clients_traites']].to_string(index=False))

2026-09-24 10:31:43,338 [INFO] ============================================================
2026-09-24 10:31:43,338 [INFO] PERFORMANCE DES PDA ET IMPACT SUR LE RECOUVREMENT
2026-09-24 10:31:43,338 [INFO] ============================================================
2026-09-24 10:31:43,402 [INFO] Leviers PDA detectes : ['Coupures', 'Envoi E-mail', 'Juridique', 'Mise en Demeure', 'Relance Avocat', 'Relance Gestion Personnalisée', 'Relance Porte à porte', 'Relance SMS Exigibilité', 'Relance Systématique', 'Relancer le client via CATO', 'Télé-Relance']
2026-09-24 10:31:43,402 [INFO] Comparaison brute traite vs non-traite (par levier)...
2026-09-24 10:31:43,902 [INFO]   /!\ Comparaison brute : biais de ciblage possible (leviers durs assignes aux clients deja les plus a risque) - voir effet ajuste ci-dessous pour un diagnostic corrige de ce biais.
2026-09-24 10:31:43,902 [INFO] Estimation de l'effet ajuste par levier (T-learner, Random Forest)...
2026-09-24 10:31:49,021 [INFO]   action_Coupur

## **PARTIE 10 : OPTIMISATION DE LA STRATEGIE DES PDA**

[NEW] Deux approches complementaires pour repondre a "comment optimiser
l'usage des PDA" :

1. **Recommandation individuelle** : a partir des effets ajustes (ITE) de
   la PARTIE 9, chaque client se voit attribuer le levier qui maximise le
   gain net estime (effet sur le taux d'encaissement, valorise en DA via
   le CA du client, moins un cout relatif du levier - couts illustratifs,
   A CALIBRER avec les couts reels de la Direction Clientele).
2. **Bandit Thompson Sampling (par segment)** : simulation par rejeu
   ("replay") des donnees historiques, qui apprend progressivement quel
   levier maximise le taux de succes (bon_payeur) au sein de chaque
   segment, sans hypothese de linearite ni de distribution. C'est la
   meme famille d'approche que l'optimisation de politique utilisee dans
   le pipeline PFE (bandit Thompson Sampling) - ici appliquee au pilotage
   des leviers de recouvrement plutot qu'aux tournees.

Les deux methodes sont complementaires : la premiere donne une
recommandation immediate et explicable par client ; la seconde montre
comment un dispositif d'apprentissage continu convergerait vers la
meilleure politique si le recouvrement pilotait ses leviers en boucle
(deploiement -> mesure -> reallocation).

In [39]:
# ==============================================================================
# PARTIE 10 : OPTIMISATION DE LA STRATEGIE DES PDA
# ==============================================================================

# Couts relatifs ILLUSTRATIFS par nature de levier (echelle arbitraire 1-5,
# a remplacer par les couts reels operationnels/juridiques de SEAAL). Sert
# uniquement a ne pas recommander systematiquement le levier le plus "lourd"
# quand un levier plus leger obtient un effet ajuste presque identique.
COUT_RELATIF_DEFAUT = {
    'sms': 1, 'email': 1, 'e-mail': 1, 'gpp': 1, 'appel': 1, 'cato': 1,
    'coupure': 3, 'coupures': 3,
    'demeure': 4, 'juridique': 5, 'avocat': 5, 'avocats': 5,
}


def _cout_relatif(levier_nom: str) -> float:
    nom = levier_nom.lower()
    for cle, cout in COUT_RELATIF_DEFAUT.items():
        if cle in nom:
            return cout
    return 2.0  # cout par defaut si le libelle ne correspond a aucun mot-cle connu


class PDAOptimizer:
    """Recommandation individuelle du levier optimal (a partir des ITE de la
    PARTIE 9) et simulation d'un bandit Thompson Sampling par segment."""

    def __init__(self, config: PipelineConfig, action_cols: List[str]):
        self.config = config
        self.action_cols = action_cols

    # ------------------------------------------------------------------
    def recommander_par_client(self, df: pd.DataFrame, poids_cout: float = 0.02) -> Tuple[pd.DataFrame, Dict]:
        """Pour chaque client, choisit le levier (ou 'Aucune action') qui
        maximise le gain net estime = gain_DA_leviers - poids_cout * cout_relatif * CA.
        `poids_cout` convertit le cout relatif (echelle 1-5) en une penalite
        exprimee dans la meme unite que le gain (DA), proportionnelle au CA
        du client - a ajuster si les couts reels sont connus."""
        logger.info("=" * 60)
        logger.info("RECOMMANDATION DE LA STRATEGIE PDA OPTIMALE (par client)")
        logger.info("=" * 60)

        ite_cols = {col: f'ite_{col}' for col in self.action_cols if f'ite_{col}' in df.columns}
        if not ite_cols:
            logger.warning("Aucun effet ajuste (ITE) disponible : executer la PARTIE 9 au prealable.")
            return df, {}

        df = df.copy()
        gains = pd.DataFrame(index=df.index)
        gains['Aucune action'] = 0.0
        for col, ite_col in ite_cols.items():
            levier_nom = col.replace('action_', '')
            cout = _cout_relatif(levier_nom)
            gain_da = df[ite_col] * df['ca_2025'].clip(lower=0)
            gains[levier_nom] = gain_da - poids_cout * cout * df['ca_2025'].clip(lower=0)

        df['levier_recommande'] = gains.idxmax(axis=1)
        df['gain_espere_DA'] = gains.max(axis=1)

        # Politique actuelle (le levier reellement recu, ou 'Aucune action')
        def levier_actuel(row):
            recus = [col.replace('action_', '') for col in ite_cols if row.get(col, 0) > 0]
            return recus[0] if recus else 'Aucune action'
        df['levier_actuel'] = df.apply(levier_actuel, axis=1)

        gain_politique_actuelle = float(sum(
            gains.loc[i, df.loc[i, 'levier_actuel']] for i in df.index
        ))
        gain_politique_optimale = float(df['gain_espere_DA'].sum())
        gain_reallocation = gain_politique_optimale - gain_politique_actuelle

        synthese = {
            'gain_politique_actuelle_DA': gain_politique_actuelle,
            'gain_politique_optimale_DA': gain_politique_optimale,
            'gain_supplementaire_reallocation_DA': gain_reallocation,
        }
        logger.info(f"Gain estime de la politique actuelle   : {gain_politique_actuelle:,.0f} DA")
        logger.info(f"Gain estime de la politique optimisee  : {gain_politique_optimale:,.0f} DA")
        logger.info(f"Gain SUPPLEMENTAIRE par reallocation    : {gain_reallocation:,.0f} DA")

        repartition = df['levier_recommande'].value_counts()
        logger.info("Repartition recommandee des leviers :\n" + repartition.to_string())

        fig, ax = plt.subplots(figsize=(9, 6))
        repartition.sort_values().plot(kind='barh', ax=ax, color='teal')
        ax.set_title("Repartition recommandee des leviers (politique optimisee)")
        ax.set_xlabel("Nombre de clients")
        plt.tight_layout()
        plt.savefig(os.path.join(self.config.output_path, 'pda_repartition_recommandee.png'), dpi=150)
        plt.close()

        cols_export = [c for c in ['code_client', 'segment_nom', 'levier_actuel', 'levier_recommande',
                                    'gain_espere_DA', 'score_risque_global', 'creance_totale']
                       if c in df.columns]
        df.sort_values('gain_espere_DA', ascending=False)[cols_export].head(500).to_csv(
            os.path.join(self.config.output_path, 'pda_recommandations_par_client.csv'), index=False
        )

        return df, synthese

    # ------------------------------------------------------------------
    def simuler_bandit_thompson(self, df: pd.DataFrame, n_rounds: int = 3000,
                                 segment_col: str = 'segment_nom',
                                 outcome_col: str = 'bon_payeur') -> pd.DataFrame:
        """Simule, par rejeu (replay) des donnees historiques, un bandit
        Thompson Sampling (Beta-Bernoulli) par segment : a chaque round, le
        bandit choisit un levier selon sa croyance courante, recoit une
        recompense binaire tiree par bootstrap parmi les clients du segment
        ayant reellement recu ce levier, puis met a jour sa croyance. Le
        bras converge vers le levier historiquement le plus associe a un
        bon comportement de paiement, sans hypothese de linearite."""
        logger.info("=" * 60)
        logger.info("SIMULATION BANDIT THOMPSON SAMPLING (par segment)")
        logger.info("=" * 60)

        if segment_col not in df.columns or outcome_col not in df.columns:
            logger.warning(f"Colonnes '{segment_col}'/'{outcome_col}' absentes : simulation ignoree.")
            return pd.DataFrame()

        rng = np.random.default_rng(RANDOM_STATE)
        bras_noms = ['Aucune action'] + [c.replace('action_', '') for c in self.action_cols]
        resultats = []
        historiques_regret = {}

        for segment, grp in df.groupby(segment_col):
            pools = {}
            pools['Aucune action'] = grp.loc[grp[[c for c in self.action_cols]].sum(axis=1) == 0, outcome_col].values
            for col in self.action_cols:
                levier_nom = col.replace('action_', '')
                pools[levier_nom] = grp.loc[grp[col] > 0, outcome_col].values

            bras_valides = [b for b in bras_noms if len(pools.get(b, [])) >= 5]
            if len(bras_valides) < 2:
                continue

            alpha = {b: 1.0 for b in bras_valides}
            beta_ = {b: 1.0 for b in bras_valides}
            meilleur_taux_reel = max(pools[b].mean() for b in bras_valides)  # oracle pour le regret
            regret_cumule, historique = 0.0, []

            for t in range(n_rounds):
                echantillons = {b: rng.beta(alpha[b], beta_[b]) for b in bras_valides}
                choix = max(echantillons, key=echantillons.get)
                recompense = rng.choice(pools[choix])
                alpha[choix] += recompense
                beta_[choix] += (1 - recompense)
                regret_cumule += meilleur_taux_reel - pools[choix].mean()
                if t % max(1, n_rounds // 100) == 0:
                    historique.append(regret_cumule)

            taux_estimes = {b: alpha[b] / (alpha[b] + beta_[b]) for b in bras_valides}
            meilleur_bras = max(taux_estimes, key=taux_estimes.get)
            resultats.append({
                'segment': segment,
                'levier_recommande_bandit': meilleur_bras,
                'taux_succes_estime': taux_estimes[meilleur_bras],
                'nb_observations_segment': len(grp),
                'nb_bras_testes': len(bras_valides),
            })
            historiques_regret[segment] = historique

        bandit_df = pd.DataFrame(resultats).sort_values('nb_observations_segment', ascending=False)
        bandit_df.to_csv(os.path.join(self.config.output_path, 'pda_bandit_recommandation_segment.csv'), index=False)

        # Courbe de convergence (regret cumule) pour les 4 segments les plus representes
        top_segments = bandit_df.head(4)['segment'].tolist()
        if top_segments:
            fig, ax = plt.subplots(figsize=(10, 6))
            for seg in top_segments:
                hist = historiques_regret.get(seg, [])
                if hist:
                    ax.plot(np.linspace(0, n_rounds, len(hist)), hist, label=str(seg))
            ax.set_xlabel("Round de simulation")
            ax.set_ylabel("Regret cumule (vs meilleur levier oracle)")
            ax.set_title("Convergence du bandit Thompson Sampling par segment")
            ax.legend(fontsize=8)
            plt.tight_layout()
            plt.savefig(os.path.join(self.config.output_path, 'pda_bandit_convergence.png'), dpi=150)
            plt.close()

        logger.info("Recommandation du bandit par segment :\n" + bandit_df.to_string(index=False))
        return bandit_df

In [40]:
pda_optimizer = PDAOptimizer(CONFIG, pda_analyzer.action_cols)
df_pda_reco, synthese_reallocation = pda_optimizer.recommander_par_client(df_pda)
bandit_recommandation = pda_optimizer.simuler_bandit_thompson(df_pda_reco)

2026-09-24 10:32:37,347 [INFO] ============================================================
2026-09-24 10:32:37,349 [INFO] RECOMMANDATION DE LA STRATEGIE PDA OPTIMALE (par client)
2026-09-24 10:32:37,349 [INFO] ============================================================
2026-09-24 10:33:06,355 [INFO] Gain estime de la politique actuelle   : -18,046,300 DA
2026-09-24 10:33:06,355 [INFO] Gain estime de la politique optimisee  : 144,588,967 DA
2026-09-24 10:33:06,355 [INFO] Gain SUPPLEMENTAIRE par reallocation    : 162,635,266 DA
2026-09-24 10:33:06,365 [INFO] Repartition recommandee des leviers :
levier_recommande
Coupures                         49895
Aucune action                    16207
Envoi E-mail                      6949
Relance Gestion Personnalisée     4561
Relance Avocat                    1797
Relancer le client via CATO       1763
Relance Systématique              1379
Relance Porte à porte              736
Relance SMS Exigibilité            642
Télé-Relance                

## **PARTIE 11 : IMPACT COMBINÉ — FACTURATION OPTIMISÉE + PDA OPTIMISÉS**

[NEW] Dernier maillon demande : quantifier l'impact CONJOINT de deux
leviers d'optimisation distincts mais complementaires :

- **Facturation optimisee** : correction des anomalies de facturation/
  encaissement detectees en PARTIE 5 (`anomalie_encaissement`,
  `anomalie_erreur_releve`) - une partie de la creance liee a ces
  anomalies redevient recouvrable une fois la facturation corrigee.
  Hypothese de taux de recuperation **explicitement parametrable**
  (30% par defaut) : ce n'est PAS une mesure mais une estimation a
  calibrer avec un echantillon reel de corrections deja effectuees.
- **PDA optimises** : gain de reallocation des leviers de recouvrement
  estime en PARTIE 10 (T-learner + recommandation par client).

Les deux gains sont presentes cote a cote ET combines, avec le detail des
hypotheses retenues, puis exportes dans un classeur Excel unique,
entierement modifiable et telechargeable.

In [41]:
# ==============================================================================
# PARTIE 11 : IMPACT COMBINE FACTURATION + PDA, ET EXPORT FINAL
# ==============================================================================

class ImpactCombineAnalyzer:
    """Quantifie et compare l'impact de l'optimisation de la facturation,
    des PDA, et de leur combinaison, sous forme de scenarios chiffres."""

    def __init__(self, config: PipelineConfig):
        self.config = config

    def analyze(self, df: pd.DataFrame, synthese_pda: Dict,
                taux_recuperation_facturation: float = 0.30) -> Tuple[pd.DataFrame, Dict]:
        logger.info("=" * 60)
        logger.info("IMPACT COMBINE : FACTURATION OPTIMISEE + PDA OPTIMISES")
        logger.info("=" * 60)

        mask_facturation = pd.Series(False, index=df.index)
        for col in ('anomalie_encaissement', 'anomalie_erreur_releve'):
            if col in df.columns:
                mask_facturation |= (df[col] == 1)

        nb_clients_facturation = int(mask_facturation.sum())
        creance_concernee = float(df.loc[mask_facturation, 'creance_totale'].sum())
        gain_facturation = creance_concernee * taux_recuperation_facturation

        gain_pda = float(synthese_pda.get('gain_supplementaire_reallocation_DA', 0.0))

        ca_total = float(df['ca_2025'].sum()) or 1.0
        creance_totale_pf = float(df['creance_totale'].sum()) or 1.0

        scenarios = pd.DataFrame([
            {'scenario': 'Statu quo (situation actuelle)', 'gain_estime_DA': 0.0},
            {'scenario': 'Facturation optimisee seule', 'gain_estime_DA': gain_facturation},
            {'scenario': 'PDA optimises seuls', 'gain_estime_DA': gain_pda},
            {'scenario': 'Combine (facturation + PDA optimises)', 'gain_estime_DA': gain_facturation + gain_pda},
        ])
        scenarios['pct_du_CA_2025'] = scenarios['gain_estime_DA'] / ca_total * 100
        scenarios['pct_de_la_creance_totale'] = scenarios['gain_estime_DA'] / creance_totale_pf * 100

        logger.info("\n" + scenarios.to_string(index=False))

        fig, ax = plt.subplots(figsize=(9, 6))
        colors = ['grey', 'steelblue', 'seagreen', 'darkorange']
        ax.bar(scenarios['scenario'], scenarios['gain_estime_DA'], color=colors)
        ax.set_ylabel("Gain estime (DA)")
        ax.set_title("Impact combine : facturation optimisee + PDA optimises")
        for i, v in enumerate(scenarios['gain_estime_DA']):
            ax.text(i, v, f"{v:,.0f}", ha='center', va='bottom', fontsize=9)
        plt.xticks(rotation=15, ha='right')
        plt.tight_layout()
        plt.savefig(os.path.join(self.config.output_path, 'impact_combine_scenarios.png'), dpi=150)
        plt.close()

        details = {
            'nb_clients_facturation_concernes': nb_clients_facturation,
            'creance_concernee_facturation_DA': creance_concernee,
            'taux_recuperation_hypothese': taux_recuperation_facturation,
            'gain_facturation_DA': gain_facturation,
            'gain_pda_DA': gain_pda,
        }
        return scenarios, details


# ==============================================================================
# EXPORT FINAL : classeur Excel unique, telechargeable, entierement modifiable
# ==============================================================================

from openpyxl.styles import Font as XlFont, PatternFill as XlFill, Alignment as XlAlign
from openpyxl.utils import get_column_letter as xl_col_letter
from openpyxl.chart import BarChart as XlBarChart, Reference as XlReference

_HEADER_FILL = XlFill("solid", fgColor="1B2A4A")
_HEADER_FONT = XlFont(color="FFFFFF", bold=True)


def _ecrire_feuille(writer, df_table: pd.DataFrame, sheet_name: str, titre: str = None, startrow: int = 0):
    startrow_data = startrow + (2 if titre else 0)
    df_table.to_excel(writer, sheet_name=sheet_name, index=False, startrow=startrow_data)
    ws = writer.sheets[sheet_name]

    if titre:
        ws.cell(row=startrow + 1, column=1, value=titre).font = XlFont(bold=True, size=13, color="1B2A4A")

    header_row = startrow_data + 1
    for j, col in enumerate(df_table.columns, start=1):
        c = ws.cell(row=header_row, column=j)
        c.fill = _HEADER_FILL
        c.font = _HEADER_FONT
        c.alignment = XlAlign(horizontal="center", vertical="center")
        max_len = max([len(str(col))] + [len(str(v)) for v in df_table[col].astype(str).head(500)])
        ws.column_dimensions[xl_col_letter(j)].width = min(max_len + 3, 40)
    return header_row


def exporter_resultats_pda_finaux(pda_naif, pda_uplift, df_pda_reco, bandit_df,
                                    scenarios, details, config: PipelineConfig):
    """Exporte l'ensemble des resultats PDA / impact combine dans un unique
    classeur Excel modifiable, telechargeable depuis le repertoire de sortie."""
    output_file = os.path.join(config.output_path, "Resultats_Optimisation_PDA_Recouvrement.xlsx")

    top_reco = df_pda_reco.sort_values('gain_espere_DA', ascending=False)
    cols_reco = [c for c in ['code_client', 'segment_nom', 'levier_actuel', 'levier_recommande',
                              'gain_espere_DA', 'score_risque_global', 'creance_totale', 'taux_encaissement']
                 if c in top_reco.columns]
    top_reco = top_reco[cols_reco].head(300)

    with pd.ExcelWriter(output_file, engine='openpyxl') as writer:
        _ecrire_feuille(writer, pda_naif, "Performance_Brute",
                         "Comparaison brute traite vs non-traite (biais de ciblage possible - voir note methodologique)")
        _ecrire_feuille(writer, pda_uplift, "Performance_Ajustee",
                         "Effet ajuste par levier (T-learner) - controle pour le profil de risque du client")
        _ecrire_feuille(writer, bandit_df, "Recommandation_Bandit",
                         "Levier recommande par segment (simulation Thompson Sampling)")
        _ecrire_feuille(writer, top_reco, "Recommandations_Client",
                         "Top 300 clients par gain espere (politique optimisee individuelle)")
        header_row = _ecrire_feuille(writer, scenarios, "Impact_Combine",
                                       "Scenarios d'impact : facturation optimisee, PDA optimises, combine")

        ws_impact = writer.sheets["Impact_Combine"]
        detail_start = header_row + len(scenarios) + 3
        ws_impact.cell(row=detail_start, column=1, value="Hypotheses retenues").font = XlFont(bold=True, size=12, color="1B2A4A")
        for i, (k, v) in enumerate(details.items(), start=1):
            ws_impact.cell(row=detail_start + i, column=1, value=k)
            ws_impact.cell(row=detail_start + i, column=2, value=round(v, 4) if isinstance(v, float) else v)

        chart = XlBarChart()
        chart.type = "col"
        chart.title = "Gain estime par scenario (DA)"
        chart.y_axis.title = "Gain estime (DA)"
        chart.style = 10
        n = len(scenarios)
        data_ref = XlReference(ws_impact, min_col=2, min_row=header_row, max_row=header_row + n)
        cats_ref = XlReference(ws_impact, min_col=1, min_row=header_row + 1, max_row=header_row + n)
        chart.add_data(data_ref, titles_from_data=True)
        chart.set_categories(cats_ref)
        chart.width, chart.height = 18, 10
        ws_impact.add_chart(chart, f"A{detail_start + len(details) + 3}")

    logger.info(f"Classeur Excel final genere : {output_file}")
    return output_file

In [42]:
impact_analyzer = ImpactCombineAnalyzer(CONFIG)
scenarios_impact, details_impact = impact_analyzer.analyze(df_pda_reco, synthese_reallocation)

fichier_resultats = exporter_resultats_pda_finaux(
    pda_naif, pda_uplift, df_pda_reco, bandit_recommandation,
    scenarios_impact, details_impact, CONFIG
)
print(f"\nFichier de resultats telechargeable : {fichier_resultats}")

2026-09-24 10:33:07,920 [INFO] ============================================================
2026-09-24 10:33:07,920 [INFO] IMPACT COMBINE : FACTURATION OPTIMISEE + PDA OPTIMISES
2026-09-24 10:33:07,925 [INFO] ============================================================
2026-09-24 10:33:07,930 [INFO] 
                             scenario  gain_estime_DA  pct_du_CA_2025  pct_de_la_creance_totale
       Statu quo (situation actuelle)    0.000000e+00        0.000000                  0.000000
          Facturation optimisee seule    3.019713e+07       11.689590                  8.529507
                  PDA optimises seuls    1.626353e+08       62.957633                 45.938104
Combine (facturation + PDA optimises)    1.928324e+08       74.647223                 54.467611
2026-09-24 10:33:08,448 [INFO] Classeur Excel final genere : /mnt/agents/output\Resultats_Optimisation_PDA_Recouvrement.xlsx

Fichier de resultats telechargeable : /mnt/agents/output\Resultats_Optimisation_PDA_Recouvre

## **Synthèse finale**

In [43]:
logger.info("=" * 70)
logger.info("PIPELINE TERMINE")
logger.info("=" * 70)
logger.info(f"Clients traites                : {len(df_segments)}")
logger.info(f"Anomalies detectees            : {df_segments['anomaly_pred_ensemble'].sum()}")
logger.info(f"  dont alertes 'Urgent'        : {(df_segments['niveau_alerte'] == 'Urgent').sum()}")
logger.info(f"Segments de recouvrement       : {df_segments['segment'].nunique()}")
logger.info(f"Zones de tournee generees      : {len(routes_optimisees)}")
if not pda_uplift.empty:
    logger.info(f"Leviers PDA analyses           : {len(pda_uplift)}")
    logger.info(f"Meilleur levier (effet ajuste) : {pda_uplift.iloc[0]['levier']} "
                f"({pda_uplift.iloc[0]['ate_global_taux_encaissement']:+.4f})")
logger.info(f"Gain PDA (reallocation)        : {synthese_reallocation.get('gain_supplementaire_reallocation_DA', 0):,.0f} DA")
logger.info(f"Gain facturation (estime)      : {details_impact.get('gain_facturation_DA', 0):,.0f} DA")
logger.info(f"Gain combine total (estime)    : "
            f"{details_impact.get('gain_facturation_DA', 0) + details_impact.get('gain_pda_DA', 0):,.0f} DA")
logger.info(f"Resultats disponibles dans     : {CONFIG.output_path}")

df_segments.to_csv(os.path.join(CONFIG.output_path, 'dataset_final_enrichi.csv'), index=False)

print("Pipeline execute avec succes. Voir le repertoire de sortie pour l'ensemble des livrables :")
print("  - Graphiques, cartes interactives, previsions, alertes, profils de segments (PARTIES 1-8)")
print("  - Performance et effet ajuste des PDA (PARTIE 9)")
print("  - Recommandations d'optimisation individuelle et bandit Thompson Sampling (PARTIE 10)")
print(f"  - Classeur Excel final telechargeable : {fichier_resultats}")

2026-09-24 10:33:08,465 [INFO] ======================================================================
2026-09-24 10:33:08,465 [INFO] PIPELINE TERMINE
2026-09-24 10:33:08,465 [INFO] ======================================================================
2026-09-24 10:33:08,465 [INFO] Clients traites                : 84035
2026-09-24 10:33:08,473 [INFO] Anomalies detectees            : 2194
2026-09-24 10:33:08,473 [INFO]   dont alertes 'Urgent'        : 0
2026-09-24 10:33:08,473 [INFO] Segments de recouvrement       : 5
2026-09-24 10:33:08,473 [INFO] Zones de tournee generees      : 5
2026-09-24 10:33:08,479 [INFO] Leviers PDA analyses           : 10
2026-09-24 10:33:08,479 [INFO] Meilleur levier (effet ajuste) : Coupures (+0.5928)
2026-09-24 10:33:08,479 [INFO] Gain PDA (reallocation)        : 162,635,266 DA
2026-09-24 10:33:08,479 [INFO] Gain facturation (estime)      : 30,197,126 DA
2026-09-24 10:33:08,479 [INFO] Gain combine total (estime)    : 192,832,393 DA
2026-09-24 10:33:08,485 [